# PV-UAD → LAGPeR-S3CLIP — E2=60 → E3=25, LAST-only, A→G / G→A / G→A+G

Notebook này chuyển **đúng pipeline TransReID-init → E2 → E3** đã dùng cho WHU/AG-ReID.v2 sang **LAGPeR-S3CLIP** (RGB-only).

**Policy để report paper:**
- E2 train cố định **60 epoch**; E3 nhận **`pvuad_last_model.pth` + prototype bank từ `pvuad_last_checkpoint.pth` của E2**.
- E3 train cố định **25 epoch**; final evaluation chỉ dùng **`pvuad_last_model.pth` của E3**.
- Intermediate checkpoint không dùng để chọn model. `EVAL_PERIOD` được đặt bằng đúng horizon (60/25), nên mỗi stage chỉ evaluate ở epoch cuối.
- Final Flip-TTA chạy trên **3 protocol LAGPeR**: A→G, G→A, G→A+G.
- Re-ranking OFF.

Notebook giữ các fix RGB/DDP đã cần cho AG-ReID.v2: single-modality view-balanced PKM sampler, trainable TransReID ViT-Base 768-D stride 16×16, ProCA/GPD/E3 refinement, và DDP single-loader evaluation không gây NCCL watchdog.

### Dataset layout
Notebook tự tìm root chứa:
`bounding_box_train`, `query_aerial`, `query_ground`, `bounding_box_test_aerial`,
`bounding_box_test_ground`, `bounding_box_gallery`.

Nó audit chặt split LAGPeR trước khi train để tránh chạy nhiều giờ trên split sai. Với bản chuẩn, kỳ vọng:
train 40,770 ảnh / 2,708 IDs; A→G query 3,046 và gallery 15,533;
G→A query 3,046 và gallery 7,717; G→A+G gallery 20,204.

> LAGPeR-S3CLIP chỉ thay ảnh bằng bản đã làm nét; filename/split phải được giữ nguyên để so sánh benchmark hợp lệ.


In [1]:
# ============================================================
# 0. SETTINGS — LAGPeR-S3CLIP, multi-session, LAST-only
# ============================================================

import time as _time
NOTEBOOK_SESSION_STARTED_UNIX = _time.time()

PIPELINE_NAME = "PVUAD_LAGPER_S3CLIP_E2_E3_TRANSREID_LAST_ONLY"
PIPELINE_TAG = "lagper_s3clip_e2_60_e3_25_transreid768_b24_lastonly_v5_metricsindentfix"

# Optional hint only. Automatic discovery under /kaggle/input is authoritative.
LAGPER_S3CLIP_INPUT = "/kaggle/input/datasets/cpkimhianh/lagper-s3clip"

# If automatic camera-view inference fails, set ALL raw aerial camera IDs here,
# e.g. [3, 6, ...]. Normally leave None.
AERIAL_CAMERA_IDS_OVERRIDE = None

SOURCE_DIR_OVERRIDE = None

# Official MSMT17 TransReID checkpoint used by the WHU winner.
TRANSREID_CHECKPOINT_OVERRIDE = None
PRETRAIN_PATH_OVERRIDE = None
RUN_TRANSREID_INIT_SMOKE_TEST = True

TRANSREID_TARGET_STRIDE = [16, 16]
TRANSREID_TARGET_EMBED_DIM = 768
USE_SOURCE_SIE = False
USE_SOURCE_JPM_LOCAL_BRANCHES = False

# Emergency / multi-session resume overrides. Normally leave None.
E2_RESUME_PATH_OVERRIDE = None
E2_COMPLETED_DIR_OVERRIDE = None
E3_RESUME_PATH_OVERRIDE = None

AUTO_RESUME = True
ALLOW_LEGACY_E2_BOOTSTRAP = False
RUN_TRAINING = True
RUN_FINAL_FLIP_TTA = True
RUN_METADATA_PAIRING_CHECK = True

# Finish E2, save the Kaggle version, then begin E3 in a fresh session.
ALWAYS_START_E3_IN_NEW_SESSION = True

# Fixed paper horizons.
E2_EPOCHS = 60
E2_GLOBAL_BATCH = 24
E2_BASE_LR = 0.008

E3_EPOCHS = 25
E3_GLOBAL_BATCH = 24
E3_BASE_LR = 1e-4
E3_HEAD_LR_MULTIPLIER = 5.0

FEATURE_PRESERVE_WEIGHT = 0.05
SCENARIO_HARD_WEIGHT = 0.15
SCENARIO_HARD_TOPK = 5
SCENARIO_HARD_MARGIN = 0.05
SCENARIO_HARD_TAU = 0.05
SCENARIO_HARD_START_EPOCH = 3

# LAGPeR has 14 ground + 7 aerial cameras across all seven scenes.
# Cell 2 verifies the inferred mapping and keeps all ground remapped IDs first.
GROUND_MAX_CAMID = 13

SESSION_HARD_STOP_HOURS = 10.5
SESSION_STOP_RESERVE_MINUTES = 15
CHECKPOINT_EVERY_MINUTES = 30
TIME_CHECK_EVERY_STEPS = 10
MIN_EVAL_REMAINING_MINUTES = 25
MIN_E3_START_REMAINING_MINUTES = 75
MIN_TTA_START_REMAINING_MINUTES = 25

NUM_GPUS = 2
NUM_WORKERS = 4
RANDOM_SEED = 1234

FINAL_PROTOCOLS = ("A2G", "G2A", "G2AG")

if E2_GLOBAL_BATCH != 24 or abs(E2_BASE_LR - 0.008) > 1e-12:
    raise ValueError("E2 must stay at global batch 24 / LR 0.008.")
if E3_GLOBAL_BATCH != 24 or abs(E3_BASE_LR - 1e-4) > 1e-12:
    raise ValueError("E3 must stay at global batch 24 / LR 1e-4.")
if TRANSREID_TARGET_STRIDE != [16, 16] or TRANSREID_TARGET_EMBED_DIM != 768:
    raise ValueError("Keep the winning UAD ViT at stride [16,16], 768-D.")
if USE_SOURCE_SIE or USE_SOURCE_JPM_LOCAL_BRANCHES:
    raise ValueError("Source SIE/JPM local branches must remain disabled.")

SESSION_DEADLINE_UNIX = NOTEBOOK_SESSION_STARTED_UNIX + 3600.0 * SESSION_HARD_STOP_HOURS

print(PIPELINE_NAME, {
    "dataset": "LAGPeR-S3CLIP",
    "E2_epochs": E2_EPOCHS,
    "E3_epochs": E3_EPOCHS,
    "checkpoint_policy": "LAST E2 -> LAST E3 -> final tests",
    "protocols": FINAL_PROTOCOLS,
    "init": "MSMT17 TransReID -> trainable UAD ViT",
    "TIR_distillation": False,
})


PVUAD_LAGPER_S3CLIP_E2_E3_TRANSREID_LAST_ONLY {'dataset': 'LAGPeR-S3CLIP', 'E2_epochs': 60, 'E3_epochs': 25, 'checkpoint_policy': 'LAST E2 -> LAST E3 -> final tests', 'protocols': ('A2G', 'G2A', 'G2AG'), 'init': 'MSMT17 TransReID -> trainable UAD ViT', 'TIR_distillation': False}


## 1. Kiểm tra môi trường

In [2]:
import base64
import csv
import hashlib
import importlib.util
import io
import json
import os
import re
import shlex
import shutil
import subprocess
import sys
import tarfile
import urllib.request
import zipfile
from collections import defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
import torch

if not torch.cuda.is_available():
    raise RuntimeError("Enable a Kaggle GPU accelerator and restart the session.")
if torch.cuda.device_count() < NUM_GPUS:
    raise RuntimeError(
        f"This run expects {NUM_GPUS} GPUs, but Kaggle exposes {torch.cuda.device_count()}. "
        "Choose GPU T4 x2 in Notebook options."
    )

for package, pip_name in [("yacs", "yacs==0.1.8"), ("timm", "timm>=0.9,<2"), ("gdown", "gdown>=5,<6")]:
    if importlib.util.find_spec(package) is None:
        subprocess.run(
            [sys.executable, "-m", "pip", "install", "-q", pip_name],
            check=True,
        )

gpu_rows = []
for index in range(NUM_GPUS):
    props = torch.cuda.get_device_properties(index)
    gpu_rows.append({
        "index": index,
        "name": props.name,
        "memory_GiB": round(props.total_memory / 2**30, 2),
    })
display(pd.DataFrame(gpu_rows))
print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("CUDA runtime:", torch.version.cuda)


,index,name,memory_GiB
0,0,Tesla T4,14.56
1,1,Tesla T4,14.56


Python: 3.12.13
PyTorch: 2.10.0+cu128
CUDA runtime: 12.8


## 2. Khóa LAGPeR-S3CLIP + audit split/camera trước khi train

Notebook chỉ chấp nhận root có đủ 6 folder trong ảnh bạn gửi. Nó đọc filename dạng `PID_Cxx_FRAME...`,
kiểm tra số ảnh/ID/camera theo benchmark, suy ra camera aerial/ground, rồi tạo camera remap:
**ground → 0..13, aerial → 14..20** để ProCA/view-balanced sampler dùng đúng view.

**Folder compatibility fix:** accepts both `bounding_box_*` and `bouding_box_*` spellings, then normalizes them to canonical loader paths.

**v3 noise-gallery fix:** this LAGPeR copy stores the 179 `PID=-1` distractors separately in `bounding_box_gallery`. The notebook now reconstructs the published A→G / G→A / G→A+G gallery sizes instead of incorrectly expecting 7,717 / 15,533 / 20,204 images inside the physical folders themselves.

**v4 camera-map fix:** raw camera numbering has a gap at C10, so a global `camera_id % 3` rule is wrong. Training cameras are grouped by their four actual 3-camera scenes; the highest camera ID in each scene is aerial. This yields train aerial cameras `C3, C6, C9, C13` and test aerial cameras `C16, C19, C22`, which correctly reconstructs the 45 aerial + 134 ground distractors.

In [3]:
# ============================================================
# 2. Resolve + audit LAGPeR-S3CLIP
# ============================================================

# Canonical split names used by the pipeline. Some public/Kaggle copies of
# LAGPeR contain the historical typo "bouding_box_*" (missing "n").
# We accept both spellings, then create canonical symlinks for the loader.
SPLIT_ALIASES = {
    "bounding_box_train": (
        "bounding_box_train",
        "bouding_box_train",
    ),
    "query_aerial": (
        "query_aerial",
    ),
    "query_ground": (
        "query_ground",
    ),
    "bounding_box_test_aerial": (
        "bounding_box_test_aerial",
        "bouding_box_test_aerial",
    ),
    "bounding_box_test_ground": (
        "bounding_box_test_ground",
        "bouding_box_test_ground",
    ),
    "bounding_box_gallery": (
        "bounding_box_gallery",
        "bouding_box_gallery",
    ),
}
IMAGE_SPLITS = {
    alias.lower()
    for aliases in SPLIT_ALIASES.values()
    for alias in aliases
}
IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
LAGPER_FILENAME_RE = re.compile(
    r"(?P<pid>-?\d+)_C(?P<camid>\d+)_(?P<frameid>\d+)",
    re.IGNORECASE,
)

# IMPORTANT: the released/Kaggle LAGPeR layout stores the 179 PID=-1
# distractor images separately in bounding_box_gallery. Therefore the two
# positive test folders are smaller than the published protocol galleries.
#
# Physical folders in this copy:
#   aerial positives  = 7,672
#   ground positives  = 15,399
#   noise/distractors = 179
#
# Published protocol galleries are reconstructed below after camera-view
# inference:
#   G->A   = 7,672 + 45 aerial noise  = 7,717
#   A->G   = 15,399 + 134 ground noise = 15,533
#   G->A+G = aerial positives + ground positives excluding the 3,046
#            ground queries + all 179 noise = 20,204
EXPECTED = {
    "bounding_box_train": {
        "images": 40770, "ids": 2708, "cams": 12, "noise": 0,
    },
    "query_aerial": {
        "images": 3046, "ids": 1523, "cams": 3, "noise": 0,
    },
    "query_ground": {
        "images": 3046, "ids": 1523, "cams": 6, "noise": 0,
    },
    "bounding_box_test_aerial": {
        "images": 7672, "ids": 1523, "cams": 3, "noise": 0,
    },
    "bounding_box_test_ground": {
        "images": 15399, "ids": 1523, "cams": 6, "noise": 0,
    },
    "bounding_box_gallery": {
        "images": 179, "ids": 0, "cams": None, "noise": 179,
    },
}
PUBLISHED_PROTOCOL_COUNTS = {
    "A2G": {"query": 3046, "gallery": 15533},
    "G2A": {"query": 3046, "gallery": 7717},
    "G2AG": {"query": 3046, "gallery": 20204},
}
REQUIRED_DIRS = tuple(EXPECTED)

def image_files(path):
    path = Path(path)
    if not path.is_dir():
        return []
    values = []
    for root, _dirs, files in os.walk(path):
        root = Path(root)
        for name in files:
            p = root / name
            if p.suffix.lower() in IMAGE_EXTS:
                values.append(p)
    return sorted(values)

def parse_lagper_filename(path):
    match = LAGPER_FILENAME_RE.search(Path(path).name)
    if match is None:
        raise ValueError(
            f"Unexpected LAGPeR filename: {Path(path).name}. "
            "Expected PID_Cxx_FRAME... format."
        )
    return (
        int(match.group("pid")),
        int(match.group("camid")),
        int(match.group("frameid")),
    )

def split_frame(directory):
    rows = []
    bad = []
    for path in image_files(directory):
        try:
            pid, camid, frameid = parse_lagper_filename(path)
            rows.append((path, pid, camid, frameid))
        except Exception as exc:
            bad.append((str(path), str(exc)))
            if len(bad) >= 20:
                break
    if bad:
        display(pd.DataFrame(bad, columns=["path", "error"]))
        raise RuntimeError(
            "LAGPeR filename parsing failed. S3CLIP must preserve the original "
            "LAGPeR filenames."
        )
    return pd.DataFrame(rows, columns=["path", "pid", "camera_original", "frameid"])

def resolve_split_path(root, canonical_name):
    root = Path(root)
    for alias in SPLIT_ALIASES[canonical_name]:
        candidate = root / alias
        if candidate.is_dir():
            return candidate
    return None

def is_lagper_root(root):
    root = Path(root)
    return (
        root.is_dir()
        and all(resolve_split_path(root, name) is not None for name in REQUIRED_DIRS)
    )

def discover_lagper_roots():
    roots = []
    def add(path):
        path = Path(path)
        if is_lagper_root(path) and path not in roots:
            roots.append(path)

    add(LAGPER_S3CLIP_INPUT)

    base = Path("/kaggle/input")
    if base.is_dir():
        for root, dirs, _files in os.walk(base):
            root = Path(root)
            lower = {d.lower(): d for d in dirs}
            if all(
                any(alias.lower() in lower for alias in SPLIT_ALIASES[name])
                for name in REQUIRED_DIRS
            ):
                add(root)
                dirs[:] = []
                continue
            # Never recurse through image folders.
            dirs[:] = [d for d in dirs if d.lower() not in IMAGE_SPLITS]
            try:
                depth = len(root.relative_to(base).parts)
            except ValueError:
                depth = 99
            if depth >= 7:
                dirs[:] = []
    return roots

roots = discover_lagper_roots()
if not roots:
    preview = []
    for root, dirs, files in os.walk("/kaggle/input"):
        root = Path(root)
        try:
            depth = len(root.relative_to("/kaggle/input").parts)
        except Exception:
            depth = 99
        if depth <= 4 and any(
            key in {d.lower() for d in dirs}
            for key in (
                "bounding_box_train", "bouding_box_train",
                "query_aerial", "query_ground",
            )
        ):
            preview.append({
                "path": str(root),
                "dirs": sorted(dirs)[:30],
                "images_here": sum(Path(n).suffix.lower() in IMAGE_EXTS for n in files),
            })
        dirs[:] = [d for d in dirs if d.lower() not in IMAGE_SPLITS]
        if depth >= 4:
            dirs[:] = []
    if preview:
        display(pd.DataFrame(preview))
    raise FileNotFoundError(
        "Cannot find LAGPeR-S3CLIP under /kaggle/input. Attach the Kaggle dataset "
        "that contains bounding_box_train/query_aerial/query_ground/"
        "bounding_box_test_aerial/bounding_box_test_ground/bounding_box_gallery."
    )

hint = Path(LAGPER_S3CLIP_INPUT)
roots.sort(
    key=lambda p: (
        int(p == hint),
        int("s3clip" in str(p).lower()),
        int("lagper" in str(p).lower()),
        -len(p.parts),
        str(p),
    ),
    reverse=True,
)
DATA_ROOT = roots[0]
print("Resolved LAGPeR-S3CLIP root:", DATA_ROOT)

SPLIT_PATHS = {
    split: resolve_split_path(DATA_ROOT, split)
    for split in REQUIRED_DIRS
}
missing_split_paths = [
    split for split, path in SPLIT_PATHS.items() if path is None
]
if missing_split_paths:
    raise RuntimeError(
        f"Resolved root but missing split aliases: {missing_split_paths}"
    )

print("Resolved split folders:")
display(pd.DataFrame([
    {
        "canonical_split": split,
        "actual_folder": path.name,
        "actual_path": str(path),
    }
    for split, path in SPLIT_PATHS.items()
]))

frames = {}
audit_rows = []
for split in REQUIRED_DIRS:
    frame = split_frame(SPLIT_PATHS[split])
    frames[split] = frame
    non_noise = frame.loc[frame["pid"] >= 0]
    row = {
        "split": split,
        "images": int(len(frame)),
        "ids_non_noise": int(non_noise["pid"].nunique()),
        "noise_images": int((frame["pid"] < 0).sum()),
        "cameras": sorted(frame["camera_original"].unique().tolist()),
        "camera_count": int(frame["camera_original"].nunique()),
    }
    audit_rows.append(row)

audit = pd.DataFrame(audit_rows)
display(audit)

# Strict logical split lock: sharpened images must preserve benchmark membership.
for row in audit_rows:
    split = row["split"]
    expected = EXPECTED[split]
    if row["images"] != expected["images"]:
        raise RuntimeError(
            f"{split}: expected {expected['images']} images, got {row['images']}. "
            "Do not train until the S3CLIP copy is confirmed to preserve LAGPeR splits."
        )
    if expected["ids"] is not None and row["ids_non_noise"] != expected["ids"]:
        raise RuntimeError(
            f"{split}: expected {expected['ids']} non-noise IDs, got {row['ids_non_noise']}."
        )
    if row["noise_images"] != expected["noise"]:
        raise RuntimeError(
            f"{split}: expected {expected['noise']} PID=-1 noise images, "
            f"got {row['noise_images']}."
        )
    if expected["cams"] is not None and row["camera_count"] != expected["cams"]:
        raise RuntimeError(
            f"{split}: expected {expected['cams']} cameras, got {row['camera_count']} "
            f"({row['cameras']})."
        )

train_frame = frames["bounding_box_train"].copy()
query_aerial_frame = frames["query_aerial"].copy()
query_ground_frame = frames["query_ground"].copy()
gallery_aerial_frame = frames["bounding_box_test_aerial"].copy()
gallery_ground_frame = frames["bounding_box_test_ground"].copy()
noise_gallery_frame = frames["bounding_box_gallery"].copy()
if not (noise_gallery_frame["pid"] < 0).all():
    raise RuntimeError(
        "bounding_box_gallery is expected to contain only PID=-1 distractors "
        "in this LAGPeR release."
    )

train_ids = set(train_frame.loc[train_frame.pid >= 0, "pid"])
qa_ids = set(query_aerial_frame.loc[query_aerial_frame.pid >= 0, "pid"])
qg_ids = set(query_ground_frame.loc[query_ground_frame.pid >= 0, "pid"])
if len(train_ids) != 2708 or len(qa_ids) != 1523 or len(qg_ids) != 1523:
    raise RuntimeError("Unexpected LAGPeR ID split after filename parsing.")
if qa_ids != qg_ids:
    raise RuntimeError("query_aerial and query_ground do not contain the same 1,523 test IDs.")
overlap = sorted(train_ids & qa_ids)
if overlap:
    raise RuntimeError(
        f"Train/test identities overlap ({len(overlap)} IDs, e.g. {overlap[:20]}). "
        "This usually means the filename parser or dataset split is wrong."
    )

def require_query_coverage(query_frame, gallery_frame, protocol):
    qids = set(query_frame.loc[query_frame.pid >= 0, "pid"])
    gids = set(gallery_frame.loc[gallery_frame.pid >= 0, "pid"])
    missing = sorted(qids - gids)
    if missing:
        raise RuntimeError(
            f"{protocol}: {len(missing)} query identities have no gallery positive: {missing[:20]}"
        )

# Positive-only folders must already cover every query identity. The PID=-1
# distractors are added to protocol galleries only after camera-view inference.
require_query_coverage(query_aerial_frame, gallery_ground_frame, "A2G positives")
require_query_coverage(query_ground_frame, gallery_aerial_frame, "G2A positives")

# ---------------- camera-view inference ----------------
test_aerial_cams = set(query_aerial_frame.camera_original) | set(gallery_aerial_frame.camera_original)
test_ground_cams = set(query_ground_frame.camera_original) | set(gallery_ground_frame.camera_original)
if len(test_aerial_cams) != 3 or len(test_ground_cams) != 6 or test_aerial_cams & test_ground_cams:
    raise RuntimeError(
        f"Unexpected test camera split: aerial={sorted(test_aerial_cams)}, "
        f"ground={sorted(test_ground_cams)}"
    )

all_cams = set()
for frame in frames.values():
    all_cams.update(map(int, frame.camera_original.unique()))
if len(all_cams) != 21:
    raise RuntimeError(f"Expected 21 total LAGPeR cameras, got {len(all_cams)}: {sorted(all_cams)}")

train_cams = set(map(int, train_frame.camera_original.unique()))

if AERIAL_CAMERA_IDS_OVERRIDE is not None:
    aerial_cams = set(map(int, AERIAL_CAMERA_IDS_OVERRIDE))
else:
    # LAGPeR uses 3 cameras per scene: two ground cameras followed by one
    # aerial camera. Camera numbering is NOT globally modulo-3 because raw
    # camera C10 is absent in this release. Therefore the old `cam % 3`
    # heuristic misclassified C1/C4/C7 as aerial and C3/C6/C9 as ground.
    #
    # Infer the four training scenes from their 12 raw camera IDs, grouped as
    # four consecutive 3-camera scene blocks. The aerial camera is the highest
    # ID in each block, exactly as confirmed by the held-out scenes:
    #   test ground  = {14,15,17,18,20,21}
    #   test aerial  = {16,19,22}
    train_sorted = sorted(train_cams)
    if len(train_sorted) != 12:
        raise RuntimeError(
            f"Expected 12 training cameras, got {len(train_sorted)}: {train_sorted}"
        )

    train_scene_groups = [
        train_sorted[i:i + 3]
        for i in range(0, len(train_sorted), 3)
    ]
    if len(train_scene_groups) != 4 or any(len(group) != 3 for group in train_scene_groups):
        raise RuntimeError(
            f"Cannot partition training cameras into four 3-camera scenes: "
            f"{train_scene_groups}"
        )
    bad_groups = [
        group for group in train_scene_groups
        if not (group[1] == group[0] + 1 and group[2] == group[1] + 1)
    ]
    if bad_groups:
        raise RuntimeError(
            "Unexpected LAGPeR training-camera scene grouping. "
            f"Expected consecutive triples, got {train_scene_groups}. "
            "Set AERIAL_CAMERA_IDS_OVERRIDE explicitly if needed."
        )

    train_aerial_inferred = {group[-1] for group in train_scene_groups}

    # Independently validate the three held-out scenes from the supplied
    # query/test folders rather than guessing their view from camera numbers.
    test_sorted = sorted(test_aerial_cams | test_ground_cams)
    test_scene_groups = [
        test_sorted[i:i + 3]
        for i in range(0, len(test_sorted), 3)
    ]
    if (
        len(test_scene_groups) != 3
        or any(len(group) != 3 for group in test_scene_groups)
        or any(
            not (group[1] == group[0] + 1 and group[2] == group[1] + 1)
            for group in test_scene_groups
        )
        or {group[-1] for group in test_scene_groups} != test_aerial_cams
    ):
        raise RuntimeError(
            "Held-out LAGPeR camera grouping is inconsistent with "
            f"query_aerial/query_ground. groups={test_scene_groups}, "
            f"aerial={sorted(test_aerial_cams)}, ground={sorted(test_ground_cams)}"
        )

    aerial_cams = train_aerial_inferred | test_aerial_cams

ground_cams = all_cams - aerial_cams
train_aerial_cams = train_cams & aerial_cams
train_ground_cams = train_cams & ground_cams

if (
    len(aerial_cams) != 7
    or len(ground_cams) != 14
    or len(train_aerial_cams) != 4
    or len(train_ground_cams) != 8
    or test_aerial_cams != (aerial_cams - train_aerial_cams)
    or not test_ground_cams.issubset(ground_cams)
    or train_aerial_cams != {3, 6, 9, 13}
    or test_aerial_cams != {16, 19, 22}
):
    raise RuntimeError(
        "Camera-view inference does not match the published 7 aerial / 14 ground "
        "and train 4 aerial / 8 ground setup.\n"
        f"all aerial={sorted(aerial_cams)}\n"
        f"train aerial={sorted(train_aerial_cams)}\n"
        f"train ground={sorted(train_ground_cams)}\n"
        f"test aerial={sorted(test_aerial_cams)}\n"
        f"test ground={sorted(test_ground_cams)}\n"
        "Set AERIAL_CAMERA_IDS_OVERRIDE if this LAGPeR copy numbers cameras differently."
    )

ground_sorted = sorted(ground_cams)
aerial_sorted = sorted(aerial_cams)
camera_remap = {
    int(raw): int(index)
    for index, raw in enumerate(ground_sorted + aerial_sorted)
}
derived_ground_max = len(ground_sorted) - 1
if derived_ground_max != GROUND_MAX_CAMID:
    raise RuntimeError(
        f"Expected GROUND_MAX_CAMID={GROUND_MAX_CAMID}, derived {derived_ground_max}."
    )

for name, frame in frames.items():
    frame["camera_remapped"] = frame["camera_original"].map(camera_remap).astype(int)
    frame["view"] = np.where(
        frame["camera_original"].isin(aerial_cams), "aerial", "ground"
    )

train_frame = frames["bounding_box_train"]

# ---------------- reconstruct the PUBLISHED protocol galleries ----------------
# SeCap's LAGPeR protocol includes PID=-1 distractors in the gallery. In this
# Kaggle copy those 179 images live in a separate bounding_box_gallery folder.
noise_gallery_frame = frames["bounding_box_gallery"].copy()
noise_aerial_frame = noise_gallery_frame.loc[
    noise_gallery_frame["view"].eq("aerial")
].copy()
noise_ground_frame = noise_gallery_frame.loc[
    noise_gallery_frame["view"].eq("ground")
].copy()

# Ground query images are a subset of bounding_box_test_ground. For G->A+G
# they must not also appear in the gallery. Match by the preserved
# (pid, raw camera, frame id) key rather than by filesystem path.
def lagper_keys(frame):
    return set(
        zip(
            frame["pid"].astype(int),
            frame["camera_original"].astype(int),
            frame["frameid"].astype(int),
        )
    )

query_ground_keys = lagper_keys(query_ground_frame)
ground_positive_keys = list(
    zip(
        gallery_ground_frame["pid"].astype(int),
        gallery_ground_frame["camera_original"].astype(int),
        gallery_ground_frame["frameid"].astype(int),
    )
)
ground_keep_mask = [key not in query_ground_keys for key in ground_positive_keys]
g2ag_ground_remaining = gallery_ground_frame.loc[ground_keep_mask].copy()

protocol_gallery_a2g = pd.concat(
    [gallery_ground_frame, noise_ground_frame],
    ignore_index=True,
)
protocol_gallery_g2a = pd.concat(
    [gallery_aerial_frame, noise_aerial_frame],
    ignore_index=True,
)
protocol_gallery_g2ag = pd.concat(
    [gallery_aerial_frame, g2ag_ground_remaining, noise_gallery_frame],
    ignore_index=True,
)

PROTOCOL_FRAMES = {
    "A2G": (query_aerial_frame, protocol_gallery_a2g),
    "G2A": (query_ground_frame, protocol_gallery_g2a),
    "G2AG": (query_ground_frame, protocol_gallery_g2ag),
}

protocol_rows = []
for protocol, (query_frame, gallery_frame) in PROTOCOL_FRAMES.items():
    expected = PUBLISHED_PROTOCOL_COUNTS[protocol]
    require_query_coverage(query_frame, gallery_frame, protocol)
    row = {
        "protocol": protocol,
        "query_images": int(len(query_frame)),
        "gallery_images": int(len(gallery_frame)),
        "gallery_noise_images": int((gallery_frame["pid"] < 0).sum()),
        "query_ids": int(query_frame.loc[query_frame.pid >= 0, "pid"].nunique()),
        "gallery_positive_ids": int(
            gallery_frame.loc[gallery_frame.pid >= 0, "pid"].nunique()
        ),
    }
    protocol_rows.append(row)
    if row["query_images"] != expected["query"]:
        raise RuntimeError(
            f"{protocol}: expected {expected['query']} query images, "
            f"got {row['query_images']}."
        )
    if row["gallery_images"] != expected["gallery"]:
        raise RuntimeError(
            f"{protocol}: expected published gallery size {expected['gallery']}, "
            f"got {row['gallery_images']}."
        )

if len(noise_aerial_frame) + len(noise_ground_frame) != 179:
    raise RuntimeError("Noise gallery view split does not sum to 179.")
if len(noise_aerial_frame) != 45 or len(noise_ground_frame) != 134:
    raise RuntimeError(
        "LAGPeR noise split must reconstruct the published gallery sizes: "
        f"expected aerial=45, ground=134, got "
        f"aerial={len(noise_aerial_frame)}, ground={len(noise_ground_frame)}. "
        f"aerial_cameras={sorted(aerial_cams)}"
    )
if len(g2ag_ground_remaining) != 15399 - 3046:
    raise RuntimeError(
        "G2AG ground gallery exclusion is inconsistent with query_ground."
    )

print("Reconstructed official protocol galleries:")
display(pd.DataFrame(protocol_rows))
print(
    "Noise split:",
    {
        "aerial_noise": len(noise_aerial_frame),
        "ground_noise": len(noise_ground_frame),
        "total_noise": len(noise_gallery_frame),
    },
)

print("Camera mapping audit:")
display(pd.DataFrame([
    {
        "view": "ground",
        "raw_cameras": ground_sorted,
        "remapped_range": f"0..{GROUND_MAX_CAMID}",
        "train_cameras": sorted(train_ground_cams),
    },
    {
        "view": "aerial",
        "raw_cameras": aerial_sorted,
        "remapped_range": f"{GROUND_MAX_CAMID+1}..{len(all_cams)-1}",
        "train_cameras": sorted(train_aerial_cams),
    },
]))

# Build a normalized writable dataset view for the loader. This deliberately
# uses canonical split names even when the Kaggle source folder is misspelled
# as "bouding_box_*".
DATA_LINK_PARENT = Path("/kaggle/working/pvuad_lagper_data")
DATA_LINK_PARENT.mkdir(parents=True, exist_ok=True)
DATA_LINK = DATA_LINK_PARENT / "LAGPeR"

if DATA_LINK.is_symlink():
    DATA_LINK.unlink()
elif DATA_LINK.exists():
    resolved = DATA_LINK.resolve()
    if not str(resolved).startswith("/kaggle/working/"):
        raise RuntimeError(f"Unsafe generated data link path: {resolved}")
    shutil.rmtree(DATA_LINK)

DATA_LINK.mkdir(parents=True, exist_ok=True)

for canonical_split, actual_path in SPLIT_PATHS.items():
    link = DATA_LINK / canonical_split
    link.symlink_to(actual_path, target_is_directory=True)

# Preserve the experiment-definition text files in the normalized view too.
for txt in sorted(DATA_ROOT.glob("exp*.txt")):
    link = DATA_LINK / txt.name
    if not link.exists():
        link.symlink_to(txt)

print("Normalized loader view:")
display(pd.DataFrame([
    {
        "canonical_split": split,
        "source_folder": SPLIT_PATHS[split].name,
        "loader_path": str(DATA_LINK / split),
    }
    for split in REQUIRED_DIRS
]))

CAMERA_MAP_PATH = DATA_LINK_PARENT / "lagper_camera_map.json"
CAMERA_MAP = {
    "dataset": "LAGPeR-S3CLIP",
    "raw_to_remapped": {str(k): v for k, v in camera_remap.items()},
    "ground_raw_cameras": ground_sorted,
    "aerial_raw_cameras": aerial_sorted,
    "ground_max_camid": GROUND_MAX_CAMID,
    "train_ground_raw_cameras": sorted(train_ground_cams),
    "train_aerial_raw_cameras": sorted(train_aerial_cams),
}
CAMERA_MAP_PATH.write_text(json.dumps(CAMERA_MAP, indent=2), encoding="utf-8")

LAGPER_TRAIN_ID_COUNT = len(train_ids)
DATASET_AUDIT = {
    "dataset": "LAGPeR-S3CLIP",
    "root": str(DATA_ROOT),
    "split_folder_mapping": {
        split: SPLIT_PATHS[split].name
        for split in REQUIRED_DIRS
    },
    "train_ids": LAGPER_TRAIN_ID_COUNT,
    "train_images": int(len(train_frame)),
    "train_cameras": int(train_frame.camera_original.nunique()),
    "physical_folder_counts": {
        split: int(len(frame)) for split, frame in frames.items()
    },
    "noise_gallery": {
        "total": int(len(noise_gallery_frame)),
        "aerial": int(len(noise_aerial_frame)),
        "ground": int(len(noise_ground_frame)),
    },
    "A2G": {
        "query_images": int(len(query_aerial_frame)),
        "query_ids": int(query_aerial_frame.loc[query_aerial_frame.pid >= 0, "pid"].nunique()),
        "gallery_images": int(len(protocol_gallery_a2g)),
        "gallery_noise_images": int((protocol_gallery_a2g.pid < 0).sum()),
    },
    "G2A": {
        "query_images": int(len(query_ground_frame)),
        "query_ids": int(query_ground_frame.loc[query_ground_frame.pid >= 0, "pid"].nunique()),
        "gallery_images": int(len(protocol_gallery_g2a)),
        "gallery_noise_images": int((protocol_gallery_g2a.pid < 0).sum()),
    },
    "G2AG": {
        "query_images": int(len(query_ground_frame)),
        "query_ids": int(query_ground_frame.loc[query_ground_frame.pid >= 0, "pid"].nunique()),
        "gallery_images": int(len(protocol_gallery_g2ag)),
        "gallery_noise_images": int((protocol_gallery_g2ag.pid < 0).sum()),
    },
    "camera_map": CAMERA_MAP,
    "checkpoint_selection": "last_only",
}
(Path("/kaggle/working") / "lagper_s3clip_dataset_audit.json").write_text(
    json.dumps(DATASET_AUDIT, indent=2), encoding="utf-8"
)

# Preserve a compact audit table.
audit.to_csv("/kaggle/working/lagper_s3clip_split_audit.csv", index=False)

# Keep previews of the supplied experiment txt files for provenance; the
# materialized image folders above are used as the actual protocol.
EXP_PREVIEW = {}
for txt in sorted(DATA_ROOT.glob("exp*.txt")):
    try:
        lines = [
            line.strip() for line in txt.read_text(encoding="utf-8", errors="replace").splitlines()
            if line.strip()
        ]
        EXP_PREVIEW[txt.name] = lines[:8]
    except Exception as exc:
        EXP_PREVIEW[txt.name] = [f"<read error: {exc}>"]
(Path("/kaggle/working") / "lagper_exp_txt_preview.json").write_text(
    json.dumps(EXP_PREVIEW, indent=2), encoding="utf-8"
)
print("Experiment txt files:", sorted(EXP_PREVIEW))
print("UAD data link:", DATA_LINK)


Resolved LAGPeR-S3CLIP root: /kaggle/input/datasets/cpkimhianh/lagper-s3clip
Resolved split folders:


,canonical_split,actual_folder,actual_path
0,bounding_box_train,bounding_box_train,/kaggle/input/datasets/cpkimhianh/lagper-s3cli...
1,query_aerial,query_aerial,/kaggle/input/datasets/cpkimhianh/lagper-s3cli...
2,query_ground,query_ground,/kaggle/input/datasets/cpkimhianh/lagper-s3cli...
3,bounding_box_test_aerial,bouding_box_test_aerial,/kaggle/input/datasets/cpkimhianh/lagper-s3cli...
4,bounding_box_test_ground,bounding_box_test_ground,/kaggle/input/datasets/cpkimhianh/lagper-s3cli...
5,bounding_box_gallery,bounding_box_gallery,/kaggle/input/datasets/cpkimhianh/lagper-s3cli...


,split,images,ids_non_noise,noise_images,cameras,camera_count
0,bounding_box_train,40770,2708,0,"[1, 2, 3, 4, 5, 6, 7, 8, 9, 11, 12, 13]",12
1,query_aerial,3046,1523,0,"[16, 19, 22]",3
2,query_ground,3046,1523,0,"[14, 15, 17, 18, 20, 21]",6
3,bounding_box_test_aerial,7672,1523,0,"[16, 19, 22]",3
4,bounding_box_test_ground,15399,1523,0,"[14, 15, 17, 18, 20, 21]",6
5,bounding_box_gallery,179,0,179,"[1, 2, 3, 4, 5, 6, 7, 8, 9, 14, 15, 16, 17, 18...",17


Reconstructed official protocol galleries:


,protocol,query_images,gallery_images,gallery_noise_images,query_ids,gallery_positive_ids
0,A2G,3046,15533,134,1523,1523
1,G2A,3046,7717,45,1523,1523
2,G2AG,3046,20204,179,1523,1523


Noise split: {'aerial_noise': 45, 'ground_noise': 134, 'total_noise': 179}
Camera mapping audit:


,view,raw_cameras,remapped_range,train_cameras
0,ground,"[1, 2, 4, 5, 7, 8, 11, 12, 14, 15, 17, 18, 20,...",0..13,"[1, 2, 4, 5, 7, 8, 11, 12]"
1,aerial,"[3, 6, 9, 13, 16, 19, 22]",14..20,"[3, 6, 9, 13]"


Normalized loader view:


,canonical_split,source_folder,loader_path
0,bounding_box_train,bounding_box_train,/kaggle/working/pvuad_lagper_data/LAGPeR/bound...
1,query_aerial,query_aerial,/kaggle/working/pvuad_lagper_data/LAGPeR/query...
2,query_ground,query_ground,/kaggle/working/pvuad_lagper_data/LAGPeR/query...
3,bounding_box_test_aerial,bouding_box_test_aerial,/kaggle/working/pvuad_lagper_data/LAGPeR/bound...
4,bounding_box_test_ground,bounding_box_test_ground,/kaggle/working/pvuad_lagper_data/LAGPeR/bound...
5,bounding_box_gallery,bounding_box_gallery,/kaggle/working/pvuad_lagper_data/LAGPeR/bound...


Experiment txt files: ['exp1_A2G.txt', 'exp2_G2A.txt', 'exp3_A2G.txt', 'exp4_G2A.txt', 'exp5_G2AG.txt', 'exp6_G2G.txt']
UAD data link: /kaggle/working/pvuad_lagper_data/LAGPeR


## 3. Pin source UAD + patch LAGPeR RGB-only + TransReID initialization

Giữ nguyên pinned UAD/TransReID path đã chạy được trên AG-ReID.v2, thêm loader LAGPeR,
single-modality view-balanced sampler và DDP single-loader eval fix. Riêng LAGPeR evaluator dùng
quy tắc standard ReID: chỉ bỏ gallery **same PID + same camera**, cần thiết cho G→A+G.

**v5 metrics fix:** the LAGPeR evaluator patch now indents the original WHU fallback `remove = ...` line under `else:` correctly. The generated `utils/metrics.py` is also compiled in-memory before being written, so this exact indentation failure cannot pass silently.

**v6 session-2 fix:** when a previous Kaggle notebook output is attached, `utils/metrics.py` is already patched. The source preparation is now idempotent: it validates and reuses an existing LAGPeR evaluator patch instead of applying the text replacement twice. The internal pipeline tag intentionally remains `v5_metricsindentfix` so the completed E2@60 checkpoint from session 1 is still eligible for automatic resume/handoff to E3.

**v7 session-2 multipatch fix:** the reused `metrics.py` from session 1 contains two legitimate LAGPeR evaluator patch blocks. v6 incorrectly rejected `sentinel_count == 2` even though the file compiled successfully. v7 validates each block structurally and accepts the expected one-or-two evaluator variants. The internal pipeline tag stays unchanged so E2@60 from session 1 remains reusable.

**v8 session-2 structural fix:** `same PID + same camera` can legitimately appear elsewhere in upstream `metrics.py`, so global occurrence counts are not used anymore. v8 validates the actual LAGPeR `if/else` blocks locally. The internal pipeline tag is unchanged so the completed E2@60 checkpoint from session 1 remains reusable.

In [4]:
PINNED_UAD_COMMIT = "3e2a07314119402586c76096e87d40420894ad30"
OFFICIAL_REPO_URL = "https://github.com/msm8976/WHU-MARS.git"
WORK_REPO = Path("/kaggle/working/WHU_MARS_official_pinned")
UAD_DIR = WORK_REPO / "CVPR26_UAD"

if WORK_REPO.exists():
    resolved = WORK_REPO.resolve()
    if not str(resolved).startswith("/kaggle/working/"):
        raise RuntimeError(f"Unsafe generated-source path: {resolved}")
    shutil.rmtree(WORK_REPO)

def attached_source_candidates(base="/kaggle/input"):
    candidates = []
    base = Path(base)
    if not base.exists():
        return candidates
    for root, dirs, _files in os.walk(base):
        root_path = Path(root)
        if root_path.name.lower() in IMAGE_SPLITS:
            dirs[:] = []
            continue
        if root_path.name != "CVPR26_UAD" or not (root_path / "train.py").is_file():
            continue
        repo_root = root_path.parent
        declared_commit = None
        manifest_path = root_path / "PVUAD_patch_manifest.json"
        if manifest_path.is_file():
            try:
                declared_commit = json.loads(
                    manifest_path.read_text(encoding="utf-8")
                ).get("base_commit")
            except Exception:
                declared_commit = None
        # Kaggle mounts attached notebook outputs read-only and with a
        # different owner. Calling git directly there can emit
        # "detected dubious ownership". Prefer the patch manifest, which was
        # written by our previous session; use git only when no manifest commit
        # is available, and explicitly mark that repo safe for this one call.
        if declared_commit is None and (repo_root / ".git").is_dir():
            try:
                declared_commit = subprocess.check_output(
                    [
                        "git",
                        "-c", f"safe.directory={repo_root}",
                        "-C", str(repo_root),
                        "rev-parse", "HEAD",
                    ],
                    text=True,
                    stderr=subprocess.DEVNULL,
                ).strip()
            except Exception:
                pass
        if declared_commit != PINNED_UAD_COMMIT:
            # Never auto-select an unverified source tree. With no pinned
            # prior output, fall back to cloning the exact official commit.
            dirs[:] = []
            continue
        score = (
            2,
            1 if manifest_path.is_file() else 0,
            1 if "notebooks" in str(repo_root).lower() else 0,
        )
        candidates.append((score, repo_root, declared_commit))
        dirs[:] = []
    return sorted(candidates, key=lambda item: (item[0], str(item[1])), reverse=True)

source = Path(SOURCE_DIR_OVERRIDE) if SOURCE_DIR_OVERRIDE else None
if source is None:
    attached_sources = attached_source_candidates()
    if attached_sources:
        print("Attached UAD source candidates:")
        display(pd.DataFrame([
            {
                "path": str(path),
                "base_commit": commit,
                "selected": index == 0,
            }
            for index, (_score, path, commit) in enumerate(attached_sources)
        ]))
        source = attached_sources[0][1]

if source is not None:
    if (source / "CVPR26_UAD" / "train.py").is_file():
        shutil.copytree(source, WORK_REPO)
    elif source.name == "CVPR26_UAD" and (source / "train.py").is_file():
        WORK_REPO.mkdir(parents=True)
        shutil.copytree(source, UAD_DIR)
    else:
        raise FileNotFoundError(
            f"Attached/override source is not a WHU-MARS source tree: {source}"
        )
    print("Reusing attached UAD source:", source)
else:
    clone_command = [
        "git", "clone", "--no-checkout", "--filter=blob:none",
        OFFICIAL_REPO_URL, str(WORK_REPO),
    ]
    try:
        subprocess.run(clone_command, check=True)
        subprocess.run(
            ["git", "-C", str(WORK_REPO), "checkout", "--detach", PINNED_UAD_COMMIT],
            check=True,
        )
    except subprocess.CalledProcessError as error:
        raise RuntimeError(
            "Cannot clone the pinned official UAD source. Enable Kaggle Internet or attach "
            "the previous notebook output/source and set SOURCE_DIR_OVERRIDE."
        ) from error

if not (UAD_DIR / "train.py").is_file():
    raise FileNotFoundError(f"Missing UAD train.py under {UAD_DIR}")
if (WORK_REPO / ".git").is_dir():
    head = subprocess.check_output(
        ["git", "-C", str(WORK_REPO), "rev-parse", "HEAD"], text=True
    ).strip()
    if head != PINNED_UAD_COMMIT:
        raise RuntimeError(f"Source commit mismatch: expected {PINNED_UAD_COMMIT}, got {head}")

OVERLAY_ARCHIVE_B64 = '...'
overlay_bytes = base64.b64decode(OVERLAY_ARCHIVE_B64)
with tarfile.open(fileobj=io.BytesIO(overlay_bytes), mode="r:gz") as archive:
    members = archive.getmembers()
    for member in members:
        destination = (UAD_DIR / member.name).resolve()
        if not str(destination).startswith(str(UAD_DIR.resolve()) + os.sep):
            raise RuntimeError(f"Unsafe overlay member: {member.name}")
    archive.extractall(UAD_DIR)

OVERLAY_HASHES = {'config/defaults.py': '839faddc24b14d21f073c0a2b588406cd24ac50651304c7d7dcdbe8d7416b89e', 'configs/PVUAD.yml': '2ce61c2c87eced2377c6b290fa5cfc14dc17511eb85dbeee4c18e6054c727440', 'datasets/agreid_v2.py': 'f2dc8f1bb9f85ae5ea7b6d2602c38ddf22bec14b935853773e337b588e27c051', 'datasets/bases.py': '7712bac25aee214f38a913cf3aad1ce4ef33b9a713d7e177ad67ec17932c016d', 'datasets/lagper.py': '18b559e7c14b86070f65cf7792684ba14861c72ff25d0499419f33faae385f4c', 'datasets/make_dataloader.py': 'e2d6a35e71f2f32d0d26da9a7a4bbb862cb34ebffe056a433989f072b07133c9', 'datasets/pvuad_sampler.py': '93d21613c90372d10fca9d4b1a0ffbb34ec1a6f4849c9f6eb10671d736261d75', 'datasets/pvuad_transforms.py': '056fc9f25aba6361d4612bcc313f0cc2d175e2b695757f6912e09fc7b5fc61c0', 'datasets/sampler_ddp.py': 'e4c455f7c7afbfac09a503a924c7b09586dbdb8eda6c88735ad1a3e37599bf7c', 'datasets/whu_mars.py': '11d3a0c501912a9887d78601df01afb6495d2f859835a6b69d2079135dba24b3', 'model/backbones/vit_pytorch.py': '723305c1a05e204b744ddb0ff7873b5e02840329e01787200173c9064ed3ad05', 'model/make_model.py': 'bcdfef10d3a75cbb25da27eb688bb6834e97baec8b4221bbef15cf9bf472e3b5', 'processor/processor.py': 'e9a5dc613a374fde37874a4bf838789c7b359ff44c2e0d051db4e7e8fb987d7c', 'train.py': '8da1552749d67272ca6d8078ac3dd8c1df4183924d7795a6e7b0355eecf18ff5'}
for relative, expected_hash in OVERLAY_HASHES.items():
    actual_hash = hashlib.sha256((UAD_DIR / relative).read_bytes()).hexdigest()
    if actual_hash != expected_hash:
        raise RuntimeError(f"Overlay hash mismatch: {relative}")

# LAGPeR uses conventional person-ReID junk removal: discard only samples
# with the SAME identity AND SAME camera as the query. The released UAD
# evaluator removes the entire same-camera gallery (WHU/SYSU-style), which is
# equivalent for A2G/G2A because views use disjoint cameras, but not for G2A+G.
# Patch eval_func at runtime so all LAGPeR protocols use the benchmark-safe rule.
METRICS_PATH = UAD_DIR / "utils" / "metrics.py"
metrics_text = METRICS_PATH.read_text(encoding="utf-8")
if "import os\n" not in metrics_text[:2000]:
    metrics_text = metrics_text.replace(
        "import numpy as np\n", "import os\nimport numpy as np\n", 1
    )

LAGPER_PATCH_SENTINEL = 'PVUAD_LAGPER_MARKET_EVAL'

if LAGPER_PATCH_SENTINEL in metrics_text:
    # Session 2 can reuse the entire source tree saved by session 1.
    # In that case metrics.py is ALREADY patched. Reapplying the textual
    # replacement would wrap the fallback `remove = ...` line a second time
    # and corrupt indentation. Validate it and leave it byte-for-byte intact.
    compile(metrics_text, str(METRICS_PATH), "exec")
    print("LAGPeR metrics patch already present; reusing it without modification.")
else:
    metric_patch_count = 0
    for old_line in (
        "        remove = (g_camids[order] == q_camid) # sysu dataset",
        "        remove = (g_camids[order] == q_camid) # sysu metric",
    ):
        if old_line in metrics_text:
            # old_line has 8 spaces in the pristine evaluator. Once wrapped
            # by if/else, its fallback copy needs 12 spaces.
            fallback_line = "    " + old_line
            new_block = (
                '        if os.environ.get("PVUAD_LAGPER_MARKET_EVAL", "0") == "1":\n'
                '            remove = (g_pids[order] == q_pid) & (g_camids[order] == q_camid)\n'
                '        else:\n'
                + fallback_line
            )
            metrics_text = metrics_text.replace(old_line, new_block, 1)
            metric_patch_count += 1

    if metric_patch_count not in (1, 2):
        raise RuntimeError(
            "Expected to patch one or two pristine UAD evaluator branches for "
            f"LAGPeR, but patched {metric_patch_count}."
        )

    # Validate generated Python before writing.
    compile(metrics_text, str(METRICS_PATH), "exec")
    METRICS_PATH.write_text(metrics_text, encoding="utf-8")
    print("Applied LAGPeR same-PID+same-camera evaluator patch.")

# Final validation applies to BOTH fresh and reused source trees.
metrics_final_text = METRICS_PATH.read_text(encoding="utf-8")
compile(metrics_final_text, str(METRICS_PATH), "exec")

# The pinned UAD metrics.py can contain more than one SYSU-style evaluator
# branch. Session 1 legitimately patched TWO such branches, so requiring
# exactly one sentinel is incorrect.
sentinel_count = metrics_final_text.count(LAGPER_PATCH_SENTINEL)
if sentinel_count < 1:
    raise RuntimeError("LAGPeR evaluator patch is missing after source preparation.")

market_line = 'if os.environ.get("PVUAD_LAGPER_MARKET_EVAL", "0") == "1":'
market_block_count = metrics_final_text.count(market_line)
if market_block_count != sentinel_count:
    raise RuntimeError(
        "LAGPeR evaluator patch sentinel/if-block count mismatch: "
        f"sentinel={sentinel_count}, if_blocks={market_block_count}."
    )
if sentinel_count not in (1, 2):
    raise RuntimeError(
        "Unexpected number of LAGPeR evaluator patch blocks: "
        f"{sentinel_count}. Expected 1 or 2."
    )

# Validate only the actual inserted LAGPeR if/else blocks. Do not compare
# against the global count of the same-PID+same-camera expression because
# upstream metrics.py can use that expression in other valid functions too.
patch_block_re = re.compile(
    r'(?m)^\s*if os\.environ\.get\("PVUAD_LAGPER_MARKET_EVAL", "0"\) == "1":\s*\n'
    r'\s*remove = \(g_pids\[order\] == q_pid\) & \(g_camids\[order\] == q_camid\)\s*\n'
    r'\s*else:\s*\n'
    r'\s*remove = \(g_camids\[order\] == q_camid\)'
)
valid_patch_blocks = patch_block_re.findall(metrics_final_text)
if len(valid_patch_blocks) != sentinel_count:
    raise RuntimeError(
        "LAGPeR evaluator patch block validation failed: "
        f"expected {sentinel_count} valid blocks, found {len(valid_patch_blocks)}."
    )

print(
    f"Validated LAGPeR evaluator patch blocks: {sentinel_count} "
    "(local structural check passed)"
)
LAGPER_METRICS_SHA256 = hashlib.sha256(METRICS_PATH.read_bytes()).hexdigest()

patch_manifest = {
    "base_repo": OFFICIAL_REPO_URL,
    "base_commit": PINNED_UAD_COMMIT,
    "overlay_variant": "lagper_s3clip_transreid_lastonly_768_v8_session2_structuralfix",
    "overlay_sha256": OVERLAY_HASHES,
    "lagper_metrics_sha256": LAGPER_METRICS_SHA256,
    "lagper_eval_rule": "remove_same_pid_and_same_camera_only",
}
(UAD_DIR / "PVUAD_patch_manifest.json").write_text(
    json.dumps(patch_manifest, indent=2), encoding="utf-8"
)
subprocess.run([sys.executable, "-m", "compileall", "-q", str(UAD_DIR)], check=True)
print("Pinned and patched source:", UAD_DIR)



TRANSREID_MODEL_NAME = "vit_transreid_msmt.pth"
TRANSREID_GDRIVE_ID = "1x6Na97ycxS0t2Dn_0iRKWe1U5ccIqASK"
TRANSREID_SOURCE_STRIDE = [12, 12]

def find_named_file_without_images(base, filename):
    base = Path(base)
    if not base.exists():
        return []
    found = []
    for root, dirs, files in os.walk(base):
        root_path = Path(root)
        if filename in files:
            found.append(root_path / filename)
        if root_path.name.lower() in IMAGE_SPLITS:
            dirs[:] = []
    return sorted(found, key=lambda item: str(item))


def torch_load_cpu(path):
    try:
        return torch.load(path, map_location="cpu", weights_only=False)
    except TypeError:
        return torch.load(path, map_location="cpu")


def unwrap_checkpoint_state(payload):
    if isinstance(payload, dict):
        if "model" in payload and isinstance(payload["model"], dict):
            payload = payload["model"]
        elif "state_dict" in payload and isinstance(payload["state_dict"], dict):
            payload = payload["state_dict"]
    if not isinstance(payload, dict):
        raise ValueError("Checkpoint does not contain a state_dict")
    return {str(key).replace("module.", ""): value for key, value in payload.items()}


TRANSREID_CACHE = Path("/kaggle/working") / TRANSREID_MODEL_NAME
checkpoint_override = TRANSREID_CHECKPOINT_OVERRIDE or PRETRAIN_PATH_OVERRIDE
if checkpoint_override:
    source_checkpoint = Path(checkpoint_override)
    if not source_checkpoint.is_file():
        raise FileNotFoundError(source_checkpoint)
    if source_checkpoint.resolve() != TRANSREID_CACHE.resolve():
        shutil.copy2(source_checkpoint, TRANSREID_CACHE)
elif TRANSREID_CACHE.is_file():
    pass
else:
    attached = find_named_file_without_images("/kaggle/input", TRANSREID_MODEL_NAME)
    if attached:
        print("Caching attached pretrained TransReID checkpoint...")
        shutil.copy2(attached[0], TRANSREID_CACHE)
    else:
        print("Downloading official MSMT17 TransReID* ViT checkpoint with gdown...")
        try:
            import gdown
            result = gdown.download(
                id=TRANSREID_GDRIVE_ID,
                output=str(TRANSREID_CACHE),
                quiet=False,
            )
            if not result:
                raise RuntimeError("gdown returned no output path")
        except Exception as error:
            raise RuntimeError(
                "Cannot obtain vit_transreid_msmt.pth. Enable Kaggle Internet, "
                "or attach the official checkpoint as a Kaggle Dataset and set "
                "TRANSREID_CHECKPOINT_OVERRIDE."
            ) from error

if not TRANSREID_CACHE.is_file():
    raise FileNotFoundError(TRANSREID_CACHE)
if TRANSREID_CACHE.stat().st_size < 300_000_000:
    raise RuntimeError(
        "TransReID checkpoint is unexpectedly small; this is likely an HTML/"
        "partial download: {} bytes".format(TRANSREID_CACHE.stat().st_size)
    )

payload = torch_load_cpu(TRANSREID_CACHE)
tr_state = unwrap_checkpoint_state(payload)
required_transreid_keys = {
    "base.cls_token",
    "base.pos_embed",
    "base.patch_embed.proj.weight",
    "base.blocks.0.attn.qkv.weight",
    "base.blocks.11.attn.qkv.weight",
    "base.norm.weight",
    # These two are validated only to make sure this is the official full
    # TransReID checkpoint; they are intentionally NOT transplanted into UAD.
    "base.sie_embed",
    "b1.0.attn.qkv.weight",
    "b2.0.attn.qkv.weight",
}
missing = sorted(required_transreid_keys - set(tr_state))
if missing:
    raise RuntimeError(
        "Attached/downloaded file is not the expected full MSMT17 TransReID "
        f"checkpoint; missing keys: {missing}"
    )
if tr_state["base.cls_token"].shape[-1] != 768:
    raise RuntimeError("Expected ViT-B hidden dimension 768.")
if tuple(tr_state["base.sie_embed"].shape) != (15, 1, 768):
    raise RuntimeError(
        "Unexpected MSMT17 source SIE shape: "
        + str(tuple(tr_state["base.sie_embed"].shape))
    )
print(
    "Validated pretrained TransReID source:",
    {
        "path": str(TRANSREID_CACHE),
        "size_MiB": round(TRANSREID_CACHE.stat().st_size / 2**20, 1),
        "source_pos_embed": tuple(tr_state["base.pos_embed"].shape),
        "source_SIE": tuple(tr_state["base.sie_embed"].shape),
        "source_has_JPM": True,
        "target_embedding_dim": 768,
        "target_stride": TRANSREID_TARGET_STRIDE,
        "target_SIE": False,
        "target_JPM": False,
        "backbone_will_be_trainable": True,
    },
)
del payload, tr_state

TRANSREID_PATH = TRANSREID_CACHE
PRETRAIN_PATH = TRANSREID_PATH
print("Persistent TransReID initialization checkpoint:", PRETRAIN_PATH)


if RUN_TRANSREID_INIT_SMOKE_TEST:
    print("Running LAGPeR-S3CLIP trainable 768-D TransReID-initialized UAD smoke test...")
    TRANSREID_INIT_AUDIT_PATH = Path(
        "/kaggle/working/transreid_trainable_init_audit.json"
    )
    smoke_script = r"""
import json
import sys
from pathlib import Path
import torch

uad_dir = Path(sys.argv[1])
checkpoint = Path(sys.argv[2])
audit_path = Path(sys.argv[3])
sys.path.insert(0, str(uad_dir))

from config import cfg
from model import make_model

def load_state(path):
    try:
        payload = torch.load(path, map_location="cpu", weights_only=False)
    except TypeError:
        payload = torch.load(path, map_location="cpu")
    if isinstance(payload, dict):
        if "model" in payload and isinstance(payload["model"], dict):
            payload = payload["model"]
        elif "state_dict" in payload and isinstance(payload["state_dict"], dict):
            payload = payload["state_dict"]
    return {str(k).replace("module.", ""): v for k, v in payload.items()}

cfg.merge_from_file(str(uad_dir / "configs/PVUAD.yml"))
cfg.MODEL.PRETRAIN_CHOICE = "transreid"
cfg.MODEL.PRETRAIN_PATH = str(checkpoint)
cfg.MODEL.STRIDE_SIZE = [16, 16]
cfg.MODEL.SIE_CAMERA = False
cfg.MODEL.SIE_VIEW = False
cfg.DATASETS.NAMES = "LAGPeR"
cfg.DATASETS.MODALITIES = ["RGB"]
cfg.PVUAD.GROUND_MAX_CAMID = 13
cfg.PVUAD.VFPROCA = True
cfg.PVUAD.LOCAL_CONSISTENCY = False
cfg.PVUAD.TIR_DISTILLATION = False

model = make_model(cfg, num_class=2708, camera_num=12, view_num=0).cuda().train()
if model.transreid_init_audit is None:
    raise RuntimeError("Missing TransReID initialization audit")
if model.in_planes != 768:
    raise RuntimeError("UAD backbone must remain 768-D")
if hasattr(model.base, "sie_embed"):
    raise RuntimeError("Source-camera SIE must not be copied into target UAD")
if not all(parameter.requires_grad for parameter in model.base.parameters()):
    raise RuntimeError("TransReID-initialized backbone is unexpectedly frozen")

source = load_state(checkpoint)
source_qkv = source["base.blocks.0.attn.qkv.weight"]
target_qkv = model.base.blocks[0].attn.qkv.weight.detach().cpu()
init_qkv_max_abs = float((source_qkv - target_qkv).abs().max())
source_global_final_qkv = source["b1.0.attn.qkv.weight"]
target_final_qkv = model.base.blocks[11].attn.qkv.weight.detach().cpu()
global_final_qkv_max_abs = float(
    (source_global_final_qkv - target_final_qkv).abs().max()
)
if init_qkv_max_abs > 1e-7 or global_final_qkv_max_abs > 1e-7:
    raise RuntimeError(
        "TransReID weights were not transplanted exactly: "
        f"shared_qkv={init_qkv_max_abs}, global_final_qkv={global_final_qkv_max_abs}"
    )

SMOKE_BATCH = 2
images = [
    torch.randn(SMOKE_BATCH, 3, 256, 128, device="cuda"),
]
output = model(x=images, mode=0)
cls_score, global_feat = output[0], output[1]
if tuple(global_feat.shape) != (SMOKE_BATCH, 768):
    raise RuntimeError(f"Unexpected UAD feature shape: {tuple(global_feat.shape)}")
loss = cls_score.float().square().mean() + 0.01 * global_feat[:, :16].float().square().mean()
loss.backward()

qkv_grad = model.base.blocks[0].attn.qkv.weight.grad
patch_grad = model.base.patch_embed.proj.weight.grad
if qkv_grad is None or patch_grad is None:
    raise RuntimeError("No gradient reached the TransReID-initialized backbone")
qkv_grad_norm = float(qkv_grad.float().norm().detach().cpu())
patch_grad_norm = float(patch_grad.float().norm().detach().cpu())
if not (qkv_grad_norm > 0 and patch_grad_norm > 0):
    raise RuntimeError(
        f"Backbone gradients must be nonzero; qkv={qkv_grad_norm}, "
        f"patch={patch_grad_norm}"
    )

audit = dict(model.transreid_init_audit)
audit.update({
    "smoke_test": "OK",
    "feature_shape": list(global_feat.shape),
    "smoke_batch_size": SMOKE_BATCH,
    "batchnorm_train_mode_checked": True,
    "init_qkv_max_abs": init_qkv_max_abs,
    "global_final_qkv_max_abs": global_final_qkv_max_abs,
    "qkv_grad_norm": qkv_grad_norm,
    "patch_grad_norm": patch_grad_norm,
    "backbone_all_trainable": True,
    "source_sie_used": False,
    "source_jpm_local_branches_used": False,
    "source_jpm_global_final_branch_used": True,
    "target_stride": [16, 16],
})
audit_path.write_text(json.dumps(audit, indent=2), encoding="utf-8")
print("TRANSREID_TRAINABLE_INIT_SMOKE_OK", audit)
"""
    smoke_env = os.environ.copy()
    smoke_env["CUDA_VISIBLE_DEVICES"] = "0"
    smoke_env["PYTHONPATH"] = str(UAD_DIR) + os.pathsep + smoke_env.get(
        "PYTHONPATH", ""
    )
    result = subprocess.run(
        [
            sys.executable,
            "-c",
            smoke_script,
            str(UAD_DIR),
            str(TRANSREID_PATH),
            str(TRANSREID_INIT_AUDIT_PATH),
        ],
        cwd=UAD_DIR,
        env=smoke_env,
        text=True,
        capture_output=True,
    )
    print(result.stdout)
    if result.returncode != 0:
        print(result.stderr)
        raise RuntimeError(
            "Trainable TransReID initialization smoke test failed. "
            "Do not start E2 until the initialization path is fixed."
        )


Attached UAD source candidates:


,path,base_commit,selected
0,/kaggle/input/notebooks/thienbao1604/pvuad-lag...,3e2a07314119402586c76096e87d40420894ad30,True


Reusing attached UAD source: /kaggle/input/notebooks/thienbao1604/pvuad-lagper-s3clip-e2-60-e3-25-last-onl6ec07fd7f2/WHU_MARS_official_pinned
LAGPeR metrics patch already present; reusing it without modification.
Validated LAGPeR evaluator patch blocks: 2 (local structural check passed)
Pinned and patched source: /kaggle/working/WHU_MARS_official_pinned/CVPR26_UAD


/tmp/ipykernel_23/3429726758.py:127: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  archive.extractall(UAD_DIR)


Caching attached pretrained TransReID checkpoint...
Validated pretrained TransReID source: {'path': '/kaggle/working/vit_transreid_msmt.pth', 'size_MiB': 399.8, 'source_pos_embed': (1, 211, 768), 'source_SIE': (15, 1, 768), 'source_has_JPM': True, 'target_embedding_dim': 768, 'target_stride': [16, 16], 'target_SIE': False, 'target_JPM': False, 'backbone_will_be_trainable': True}
Persistent TransReID initialization checkpoint: /kaggle/working/vit_transreid_msmt.pth
Running LAGPeR-S3CLIP trainable 768-D TransReID-initialized UAD smoke test...
using Transformer_type: vit_base_in as a backbone
using stride: [16, 16], and patch number is num_y16 * num_x8
TransReID -> trainable UAD backbone initialization: {'checkpoint': '/kaggle/working/vit_transreid_msmt.pth', 'loaded_tensor_count': 152, 'target_tensor_count': 152, 'mismatch_count': 0, 'resized_pos_embed': {'source_shape': [1, 211, 768], 'target_shape': [1, 129, 768], 'target_grid': [16, 8]}, 'source_sie_loaded': False, 'source_jpm_local_b

## 4. View audit + multi-session state machine

Notebook audit số identity có cả ground/aerial trong train set rồi tự chọn:
**E3 resume → E2 completed → E2 resume → E2 fresh**.

Pipeline tag mới hoàn toàn tách khỏi output WHU-MARS và reranking cũ, nên sẽ
không resume nhầm checkpoint dataset khác.

In [5]:
FILENAME_RE = LAGPER_FILENAME_RE

PIPELINE_ROOT = Path("/kaggle/working/pvuad_runs") / PIPELINE_TAG
E2_RUN_DIR = PIPELINE_ROOT / "e2_final"
E3_RUN_DIR = PIPELINE_ROOT / "e3_tdh"
PIPELINE_ROOT.mkdir(parents=True, exist_ok=True)
E2_RUN_DIR.mkdir(parents=True, exist_ok=True)
E3_RUN_DIR.mkdir(parents=True, exist_ok=True)
PIPELINE_STATE_PATH = PIPELINE_ROOT / "pipeline_state.json"

INIT_AUDIT_SOURCE = Path("/kaggle/working/transreid_trainable_init_audit.json")
INIT_AUDIT_DEST = PIPELINE_ROOT / "transreid_trainable_init_audit.json"
if INIT_AUDIT_SOURCE.is_file() and not INIT_AUDIT_DEST.is_file():
    shutil.copy2(INIT_AUDIT_SOURCE, INIT_AUDIT_DEST)


def read_json(path):
    path = Path(path)
    if not path.is_file():
        return {}
    try:
        return json.loads(path.read_text(encoding="utf-8"))
    except Exception as error:
        print("Ignoring unreadable JSON:", path, error)
        return {}


def write_pipeline_state(status, active_stage, **extra):
    payload = {
        "pipeline": PIPELINE_NAME,
        "pipeline_tag": PIPELINE_TAG,
        "status": status,
        "active_stage": active_stage,
        "updated_at_unix": _time.time(),
        "notebook_started_unix": NOTEBOOK_SESSION_STARTED_UNIX,
        "session_deadline_unix": SESSION_DEADLINE_UNIX,
        "remaining_minutes": max(
            0.0, (SESSION_DEADLINE_UNIX - _time.time()) / 60.0
        ),
        **extra,
    }
    temporary = PIPELINE_STATE_PATH.with_suffix(".json.tmp")
    temporary.write_text(json.dumps(payload, indent=2), encoding="utf-8")
    os.replace(temporary, PIPELINE_STATE_PATH)
    return payload


def remaining_minutes():
    return max(0.0, (SESSION_DEADLINE_UNIX - _time.time()) / 60.0)


def copy_if_present(source_dir, destination_dir, filenames):
    if source_dir is None:
        return
    source_dir = Path(source_dir)
    destination_dir = Path(destination_dir)
    destination_dir.mkdir(parents=True, exist_ok=True)
    for filename in filenames:
        source_path = source_dir / filename
        destination_path = destination_dir / filename
        if source_path.is_file() and source_path.resolve() != destination_path.resolve():
            if not destination_path.exists():
                print("Preserving:", filename)
                shutil.copy2(source_path, destination_path)


def status_progress(status):
    return status.get("progress") or {}


def candidate_sort_key(candidate):
    status = candidate["status"]
    progress = status_progress(status)
    path_text = str(candidate["root"]).lower()
    return (
        int(PIPELINE_TAG.lower() in path_text),
        int(status.get("status") == "completed"),
        int(progress.get("epoch") or 0),
        int(bool(progress.get("epoch_complete", False))),
        int(progress.get("next_iteration") or 0),
        int(status.get("global_step") or 0),
        float(status.get("saved_at_unix") or candidate["checkpoint"].stat().st_mtime),
    )


def scan_stage_candidates(base, experiment):
    base = Path(base)
    if not base.exists():
        return []
    candidates = []
    for root, dirs, files in os.walk(base):
        root_path = Path(root)
        if root_path.name.lower() in IMAGE_SPLITS:
            dirs[:] = []
            continue
        if "pvuad_last_checkpoint.pth" not in files:
            continue
        checkpoint = root_path / "pvuad_last_checkpoint.pth"
        status = read_json(root_path / "session_status.json")
        summary = read_json(root_path / "training_summary.json")
        manifest = read_json(root_path / "run_manifest.json")
        declared = status.get("experiment", summary.get("experiment"))
        lowered = str(root_path).lower()

        # This LAGPeR pipeline must never silently bootstrap from WHU/AG outputs.
        manifest_tag = manifest.get("pipeline_tag")
        manifest_dataset = manifest.get("dataset_variant")
        if manifest_tag not in (None, PIPELINE_TAG):
            continue
        if manifest_dataset not in (None, "LAGPeR-S3CLIP"):
            continue
        if manifest_tag is None and PIPELINE_TAG.lower() not in lowered:
            continue
        legacy_match = (
            experiment == "E2" and "final_e2" in lowered
        ) or (
            experiment == "E3-TDH" and "e3_tdh" in lowered
        )
        if declared != experiment and not legacy_match:
            continue
        if status.get("stage", summary.get("stage")) not in (None, "final"):
            continue
        # Reject a manifest that clearly belongs to a different batch/LR
        # recipe. Missing legacy fields are tolerated, but contradictory
        # fields are not.
        recipe = manifest.get("recipe") or manifest.get("effective_training") or {}
        expected = (
            {"global_batch": E2_GLOBAL_BATCH, "epochs": E2_EPOCHS}
            if experiment == "E2"
            else {"global_batch": E3_GLOBAL_BATCH, "epochs": E3_EPOCHS}
        )
        incompatible = any(
            key in recipe and int(recipe[key]) != int(value)
            for key, value in expected.items()
        )
        lr_value = recipe.get("base_lr", recipe.get("backbone_lr"))
        expected_lr = E2_BASE_LR if experiment == "E2" else E3_BASE_LR
        if lr_value is not None and abs(float(lr_value) - expected_lr) > 1e-12:
            incompatible = True
        if incompatible:
            print("Skipping incompatible stage candidate:", root_path)
            continue
        candidates.append({
            "root": root_path,
            "checkpoint": checkpoint,
            "model": root_path / "pvuad_best_model.pth",
            "last_model": root_path / "pvuad_last_model.pth",
            "status": status,
            "summary": summary,
            "manifest": manifest,
        })
        dirs[:] = []
    return sorted(candidates, key=candidate_sort_key, reverse=True)


def candidate_table(candidates, label):
    if not candidates:
        return
    print(label)
    display(pd.DataFrame([{
        "root": str(item["root"]),
        "status": item["status"].get("status", "legacy_checkpoint"),
        "epoch": status_progress(item["status"]).get("epoch"),
        "next_iteration": status_progress(item["status"]).get("next_iteration"),
        "global_step": item["status"].get("global_step"),
        "best_mAP": item["status"].get("best_mAP_percent"),
        "selected": index == 0,
    } for index, item in enumerate(candidates)]))



# View-coverage audit is invariant across sessions.
PAIRING_AUDIT_PATH = PIPELINE_ROOT / "pairing_audit.csv"
pairing_frame = None

if RUN_METADATA_PAIRING_CHECK:
    by_pid = defaultdict(lambda: {"ground": 0, "aerial": 0})
    for row in train_frame.itertuples():
        view = str(row.view)
        if view not in {"ground", "aerial"}:
            raise RuntimeError(f"Unknown LAGPeR view in audit: {view}")
        by_pid[int(row.pid)][view] += 1

    pairing_rows = []
    for pid in sorted(by_pid):
        ground = int(by_pid[pid]["ground"])
        aerial = int(by_pid[pid]["aerial"])
        pairing_rows.append({
            "pid": int(pid),
            "paired_frames": int(ground + aerial),
            "ground": ground,
            "aerial": aerial,
            "view_coverage": (
                "Ground+Aerial" if ground > 0 and aerial > 0
                else "Ground-only" if ground > 0
                else "Aerial-only" if aerial > 0
                else "No sample"
            ),
        })

    pairing_frame = pd.DataFrame(pairing_rows)
    if len(pairing_frame) != LAGPER_TRAIN_ID_COUNT:
        raise RuntimeError(
            f"Expected {LAGPER_TRAIN_ID_COUNT} train identities in LAGPeR view audit, "
            f"got {len(pairing_frame)}"
        )
    pairing_frame.to_csv(PAIRING_AUDIT_PATH, index=False)
    print("LAGPeR-S3CLIP train view coverage:")
    display(
        pairing_frame["view_coverage"]
        .value_counts(dropna=False)
        .rename_axis("view_coverage")
        .reset_index(name="identities")
    )


if E3_RESUME_PATH_OVERRIDE:
    E3_CANDIDATES = [{
        "root": Path(E3_RESUME_PATH_OVERRIDE).parent,
        "checkpoint": Path(E3_RESUME_PATH_OVERRIDE),
        "model": Path(E3_RESUME_PATH_OVERRIDE).parent / "pvuad_best_model.pth",
        "last_model": Path(E3_RESUME_PATH_OVERRIDE).parent / "pvuad_last_model.pth",
        "status": read_json(Path(E3_RESUME_PATH_OVERRIDE).parent / "session_status.json"),
        "summary": {},
        "manifest": read_json(Path(E3_RESUME_PATH_OVERRIDE).parent / "run_manifest.json"),
    }]
else:
    E3_CANDIDATES = scan_stage_candidates(E3_RUN_DIR, "E3-TDH")
    if AUTO_RESUME:
        external_e3 = scan_stage_candidates("/kaggle/input", "E3-TDH")
        external_e3 = [
            item for item in external_e3
            if PIPELINE_TAG.lower() in str(item["root"]).lower()
        ]
        E3_CANDIDATES += external_e3
        unique = {str(item["checkpoint"]): item for item in E3_CANDIDATES}
        E3_CANDIDATES = sorted(unique.values(), key=candidate_sort_key, reverse=True)

if E2_RESUME_PATH_OVERRIDE:
    E2_CANDIDATES = [{
        "root": Path(E2_RESUME_PATH_OVERRIDE).parent,
        "checkpoint": Path(E2_RESUME_PATH_OVERRIDE),
        "model": Path(E2_RESUME_PATH_OVERRIDE).parent / "pvuad_best_model.pth",
        "last_model": Path(E2_RESUME_PATH_OVERRIDE).parent / "pvuad_last_model.pth",
        "status": read_json(Path(E2_RESUME_PATH_OVERRIDE).parent / "session_status.json"),
        "summary": {},
        "manifest": read_json(Path(E2_RESUME_PATH_OVERRIDE).parent / "run_manifest.json"),
    }]
else:
    E2_CANDIDATES = scan_stage_candidates(E2_RUN_DIR, "E2")
    if AUTO_RESUME:
        external_e2 = scan_stage_candidates("/kaggle/input", "E2")
        if not ALLOW_LEGACY_E2_BOOTSTRAP:
            external_e2 = [
                item for item in external_e2
                if PIPELINE_TAG.lower() in str(item["root"]).lower()
            ]
        E2_CANDIDATES += external_e2
        unique = {str(item["checkpoint"]): item for item in E2_CANDIDATES}
        E2_CANDIDATES = sorted(unique.values(), key=candidate_sort_key, reverse=True)

if E2_COMPLETED_DIR_OVERRIDE:
    override_root = Path(E2_COMPLETED_DIR_OVERRIDE)
    E2_CANDIDATES.insert(0, {
        "root": override_root,
        "checkpoint": override_root / "pvuad_last_checkpoint.pth",
        "model": override_root / "pvuad_best_model.pth",
        "last_model": override_root / "pvuad_last_model.pth",
        "status": read_json(override_root / "session_status.json"),
        "summary": read_json(override_root / "training_summary.json"),
        "manifest": read_json(override_root / "run_manifest.json"),
    })

candidate_table(E3_CANDIDATES, "E3 resume candidates (highest priority):")
candidate_table(E2_CANDIDATES, "E2 candidates:")

SELECTED_E3 = E3_CANDIDATES[0] if E3_CANDIDATES else None
SELECTED_E2 = E2_CANDIDATES[0] if E2_CANDIDATES else None

# Preserve compact E2 evidence even when a completed legacy E2 is used only as
# the warm-start for E3. Large model/checkpoint files stay at their selected
# input path until E3 has created its self-contained teacher/resume artifacts.
if SELECTED_E2 is not None:
    copy_if_present(
        SELECTED_E2["root"], E2_RUN_DIR,
        [
            "metrics_history.csv", "session_status.json", "training_summary.json",
            "pvuad_last_model.pth", "pvuad_best_model.pth",
            "pairing_audit.csv", "ablation_summary.csv", "effective_command.txt",
        ],
    )

if SELECTED_E3 is not None:
    ACTIVE_STAGE = "E3"
elif SELECTED_E2 is not None and SELECTED_E2["status"].get("status") == "completed":
    ACTIVE_STAGE = "E3"
elif SELECTED_E2 is not None:
    ACTIVE_STAGE = "E2"
else:
    ACTIVE_STAGE = "E2"

print("Selected pipeline stage:", ACTIVE_STAGE)
write_pipeline_state(
    "ready", ACTIVE_STAGE,
    selected_e2=str(SELECTED_E2["root"]) if SELECTED_E2 else None,
    selected_e3=str(SELECTED_E3["root"]) if SELECTED_E3 else None,
)

(PIPELINE_ROOT / "RESUME_NEXT_SESSION.txt").write_text(
    "PV-UAD LAGPeR-S3CLIP TransReID-init E2 -> E3 LAST-only multi-session pipeline\n\n"
    "1. A partial session ends normally with e2_paused or e3_paused.\n"
    "2. Start every run with Save Version -> Save & Run All; Kaggle then commits "
    "the output automatically after this notebook exits cleanly.\n"
    "3. In a new session, add the immediately preceding version output as Input.\n"
    "4. Use this exact notebook, keep Settings unchanged, and Run All.\n"
    "5. The state machine resumes E3 first, otherwise completed/partial E2.\n\n"
    f"Pipeline folder: pvuad_runs/{PIPELINE_TAG}\n",
    encoding="utf-8",
)
print("Pipeline output:", PIPELINE_ROOT)


LAGPeR-S3CLIP train view coverage:


,view_coverage,identities
0,Ground+Aerial,2708


E2 candidates:


,root,status,epoch,next_iteration,global_step,best_mAP,selected
0,/kaggle/input/notebooks/thienbao1604/pvuad-lag...,completed,60,0,136860,26.982922,True


Preserving: metrics_history.csv
Preserving: session_status.json
Preserving: training_summary.json
Preserving: pvuad_last_model.pth
Preserving: pvuad_best_model.pth
Preserving: effective_command.txt
Selected pipeline stage: E3
Pipeline output: /kaggle/working/pvuad_runs/lagper_s3clip_e2_60_e3_25_transreid768_b24_lastonly_v5_metricsindentfix


## 5. Train E2=60 → E3=25 → LAST E3 evaluate 3 LAGPeR protocols

Training chỉ dùng protocol A→G cho epoch-end diagnostic (chỉ epoch cuối). Sau E3, **cùng một last model**
được test bằng Flip-TTA trên A→G, G→A và G→A+G. Nếu hết giờ Kaggle giữa các protocol,
save output rồi Run All phiên sau; training sẽ được skip và chỉ chạy TTA còn thiếu.

In [6]:
environment = os.environ.copy()
environment.update({
    "CUDA_VISIBLE_DEVICES": ",".join(map(str, range(NUM_GPUS))),
    "PYTHONPATH": str(UAD_DIR) + os.pathsep + environment.get("PYTHONPATH", ""),
    "PYTHONUNBUFFERED": "1",
    "OMP_NUM_THREADS": "2",
    "PYTORCH_CUDA_ALLOC_CONF": "expandable_segments:True",
    "TORCH_NCCL_ASYNC_ERROR_HANDLING": "1",
    # End-of-stage training evaluation is A->G. Final tests override protocol.
    "PVUAD_LAGPER_PROTOCOL": "A2G",
    "PVUAD_LAGPER_MARKET_EVAL": "1",
})


def stage_status(run_dir):
    return read_json(Path(run_dir) / "session_status.json")


def display_stage_status(name, status):
    progress = status_progress(status)
    display(pd.DataFrame([{
        "stage": name,
        "status": status.get("status"),
        "reason": status.get("reason"),
        "epoch": progress.get("epoch"),
        "next_iteration": progress.get("next_iteration"),
        "epoch_complete": progress.get("epoch_complete"),
        "global_step": status.get("global_step"),
        "best_mAP": status.get("best_mAP_percent"),
        "remaining_minutes": round(remaining_minutes(), 1),
    }]))


def run_command_checked(command, run_dir, stage_name):
    command_text = shlex.join(command)
    (Path(run_dir) / "effective_command.txt").write_text(
        command_text + "\n", encoding="utf-8"
    )
    print("\nLaunching", stage_name)
    print(command_text)
    if not RUN_TRAINING:
        print("RUN_TRAINING=False — command prepared but not launched.")
        return {}
    result = subprocess.run(command, cwd=UAD_DIR, env=environment, check=False)
    if result.returncode != 0:
        # A DDP/NCCL process can abort after a periodic checkpoint has already
        # been written. Treat that as a resumable session rather than throwing
        # away hours of work. A missing/invalid checkpoint still fails hard.
        checkpoint = Path(run_dir) / "pvuad_last_checkpoint.pth"
        status_path = Path(run_dir) / "session_status.json"
        recovered_status = stage_status(run_dir) if status_path.is_file() else {}
        if (
            checkpoint.is_file()
            and recovered_status.get("status")
            in {"running_checkpoint", "paused_time_limit"}
        ):
            print(
                f"{stage_name} subprocess exited with code {result.returncode}, "
                "but a valid resumable checkpoint exists. "
                "Ending this notebook cleanly so the next Kaggle session can resume."
            )
            write_pipeline_state(
                "recoverable_stage_crash", stage_name,
                returncode=result.returncode,
                run_dir=str(run_dir),
                checkpoint=str(checkpoint),
            )
            display_stage_status(stage_name, recovered_status)
            return recovered_status
        write_pipeline_state(
            "failed", stage_name,
            returncode=result.returncode,
            run_dir=str(run_dir),
        )
        raise subprocess.CalledProcessError(result.returncode, command)
    required = [
        Path(run_dir) / "pvuad_last_checkpoint.pth",
        Path(run_dir) / "session_status.json",
    ]
    missing = [str(path) for path in required if not path.is_file()]
    if missing:
        raise RuntimeError(f"{stage_name} ended without required outputs: {missing}")
    status = stage_status(run_dir)
    if status.get("status") not in {
        "paused_time_limit", "completed", "running_checkpoint"
    }:
        raise RuntimeError(f"Unexpected {stage_name} status: {status}")
    display_stage_status(stage_name, status)
    return status


def make_stage_manifest(run_dir, stage_name, recipe, resume_path=None, **extra):
    payload = {
        "pipeline": PIPELINE_NAME,
        "pipeline_tag": PIPELINE_TAG,
        "stage_name": stage_name,
        "recipe": recipe,
        "dataset_variant": "LAGPeR-S3CLIP",
        "dataset_protocol": "A2G_training_eval__final_A2G_G2A_G2AG",
        "dataset_root": str(DATA_ROOT),
        "dataset_counts": DATASET_AUDIT,
        "source_commit": PINNED_UAD_COMMIT,
        "overlay_sha256": OVERLAY_HASHES,
        "resume_path": str(resume_path) if resume_path else None,
        "python": sys.version,
        "torch": torch.__version__,
        "cuda": torch.version.cuda,
        "gpus": gpu_rows,
        "seed": RANDOM_SEED,
        "session_guard": {
            "notebook_started_unix": NOTEBOOK_SESSION_STARTED_UNIX,
            "session_deadline_unix": SESSION_DEADLINE_UNIX,
            "hard_stop_hours": SESSION_HARD_STOP_HOURS,
            "stop_reserve_minutes": SESSION_STOP_RESERVE_MINUTES,
            "checkpoint_every_minutes": CHECKPOINT_EVERY_MINUTES,
            "time_check_every_steps": TIME_CHECK_EVERY_STEPS,
            "min_eval_remaining_minutes": MIN_EVAL_REMAINING_MINUTES,
        },
        **extra,
    }
    manifest_path = Path(run_dir) / "run_manifest.json"
    manifest_path.write_text(json.dumps(payload, indent=2), encoding="utf-8")
    shutil.copy2(
        UAD_DIR / "PVUAD_patch_manifest.json",
        Path(run_dir) / "PVUAD_patch_manifest.json",
    )
    return manifest_path


def e2_command(resume_path, strict_resume=True):
    # Fresh E2 receives pretrained TransReID backbone initialization exactly
    # once. Resume sessions reconstruct the model from the saved UAD checkpoint,
    # so they must NOT apply the source initialization again.
    pretrain_choice = "none" if resume_path else "transreid"
    recipe = {
        "experiment": "E2", "stage": "final", "epochs": E2_EPOCHS,
        "global_batch": E2_GLOBAL_BATCH, "base_lr": E2_BASE_LR,
        "view_balanced_sampler": True, "vfproca": True,
        "local_consistency": False, "hard_gpd": False,
        "dataset": "LAGPeR-S3CLIP", "protocol": "A2G_training_eval",
        "initialization": "MSMT17_TransReID_trainable_backbone",
        "pretrain_choice_this_session": pretrain_choice,
        "embedding_dim": 768,
        "backbone_stride": [16, 16],
        "source_sie_used": False,
        "source_jpm_local_branches_used": False,
        "source_jpm_global_final_branch_used": True,
    }
    manifest_path = make_stage_manifest(
        E2_RUN_DIR, "E2", recipe, resume_path=resume_path
    )
    opts = [
        "MODEL.DIST_TRAIN", "True",
        "MODEL.PRETRAIN_CHOICE", pretrain_choice,
        "MODEL.PRETRAIN_PATH", str(PRETRAIN_PATH),
        "MODEL.STRIDE_SIZE", "[16, 16]",
        "MODEL.SIE_CAMERA", "False",
        "MODEL.SIE_VIEW", "False",
        "DATASETS.NAMES", "LAGPeR",
        "DATASETS.MODALITIES", "['RGB']",
        "DATASETS.ROOT_DIR", str(DATA_LINK_PARENT),
        "DATALOADER.NUM_WORKERS", str(NUM_WORKERS),
        "SOLVER.SEED", str(RANDOM_SEED),
        "SOLVER.MAX_EPOCHS", str(E2_EPOCHS),
        "SOLVER.IMS_PER_BATCH", str(E2_GLOBAL_BATCH),
        "SOLVER.BASE_LR", str(E2_BASE_LR),
        "SOLVER.EVAL_PERIOD", str(E2_EPOCHS),
        "SOLVER.CHECKPOINT_PERIOD", "5",
        "TEST.IMS_PER_BATCH", "256",
        "TEST.RE_RANKING", "False",
        "TEST.TOP_K_EVAL", "0",
        "OUTPUT_DIR", str(E2_RUN_DIR),
        "PVUAD.EXPERIMENT", "E2",
        "PVUAD.STAGE", "final",
        "PVUAD.DEV_MODE", "False",
        "PVUAD.PAIRED_VIEW_SAMPLER", "True",
        "PVUAD.STRICT_PAIRED_DATA", "False",
        "PVUAD.GROUND_MAX_CAMID", str(GROUND_MAX_CAMID),
        "PVUAD.VFPROCA", "True",
        "MODEL.PCA_LOSS_WEIGHT", "0.01",
        "PVUAD.VIEW_ALIGNMENT_WEIGHT", "0.5",
        "PVUAD.INVARIANT_ID_WEIGHT", "0.5",
        "PVUAD.SCENARIO_CLS_WEIGHT", "0.1",
        "PVUAD.TEST_INVARIANT_BLEND", "0.5",
        "PVUAD.LOCAL_CONSISTENCY", "False",
        "PVUAD.SYNC_PAIRED_AUG", "False",
        "PVUAD.HARD_GPD", "False",
        "PVUAD.FINETUNE", "False",
        "PVUAD.TIR_DISTILLATION", "False",
        "PVUAD.FEATURE_PRESERVE_WEIGHT", "0.0",
        "PVUAD.SCENARIO_HARD_GPD", "False",
        "PVUAD.FINAL_FLIP_TTA", "False",
        "PVUAD.SAVE_LAST_EVERY", "5",
        "PVUAD.RUN_MANIFEST", str(manifest_path),
        "PVUAD.SESSION_DEADLINE_UNIX", str(float(SESSION_DEADLINE_UNIX)),
        "PVUAD.STOP_RESERVE_MINUTES", str(float(SESSION_STOP_RESERVE_MINUTES)),
        "PVUAD.CHECKPOINT_EVERY_MINUTES", str(float(CHECKPOINT_EVERY_MINUTES)),
        "PVUAD.TIME_CHECK_EVERY_STEPS", str(int(TIME_CHECK_EVERY_STEPS)),
        "PVUAD.MIN_EVAL_REMAINING_MINUTES", str(float(MIN_EVAL_REMAINING_MINUTES)),
        "PVUAD.STRICT_RESUME_CONFIG", str(bool(strict_resume)),
    ]
    if resume_path:
        opts += ["PVUAD.RESUME_PATH", str(resume_path)]
    return [
        sys.executable, "-m", "torch.distributed.run", "--standalone",
        f"--nproc_per_node={NUM_GPUS}", "train.py", "--config_file",
        "configs/PVUAD.yml", *opts,
    ]


E2_STATUS = SELECTED_E2["status"] if SELECTED_E2 else {}
E2_RESUME_PATH = None
E2_MODEL_PATH = None
E2_CHECKPOINT_PATH = None
E2_STRICT_RESUME = True
E2_COMPLETED_THIS_SESSION = False

if ACTIVE_STAGE == "E2":
    if SELECTED_E2 is not None:
        E2_RESUME_PATH = SELECTED_E2["checkpoint"]
        selected_hashes = (
            SELECTED_E2.get("manifest", {}).get("overlay_sha256") or {}
        )
        selected_processor_hash = selected_hashes.get("processor/processor.py")
        current_processor_hash = OVERLAY_HASHES.get("processor/processor.py")
        # E2 v2.3 checkpoints predate E3's gated fields in the resume
        # signature. Their known E2 structure is compatible; disable only the
        # signature-key comparison while retaining strict state_dict/optimizer
        # structural loading. New combined checkpoints remain fully strict.
        if (
            selected_processor_hash
            and selected_processor_hash != current_processor_hash
        ):
            E2_STRICT_RESUME = False
            print(
                "Known legacy E2 overlay detected; structural resume checks "
                "remain enabled, signature-only strictness is relaxed."
            )
        copy_if_present(
            SELECTED_E2["root"], E2_RUN_DIR,
            [
                "metrics_history.csv", "pvuad_best_model.pth", "pvuad_last_model.pth",
                "pairing_audit.csv", "ablation_summary.csv",
                "training_summary.json",
            ],
        )
    else:
        print("No checkpoint found: starting E2 from pretrained TransReID ViT weights; backbone is TRAINABLE.")

    write_pipeline_state(
        "e2_running", "E2",
        resume_path=str(E2_RESUME_PATH) if E2_RESUME_PATH else None,
    )
    E2_STATUS = run_command_checked(
        e2_command(E2_RESUME_PATH, E2_STRICT_RESUME), E2_RUN_DIR, "E2"
    )
    if not RUN_TRAINING:
        write_pipeline_state("dry_run", "E2")
    elif E2_STATUS.get("status") in {"paused_time_limit", "running_checkpoint"}:
        write_pipeline_state(
            "e2_paused", "E2", e2_status=E2_STATUS,
            next_action="Attach this output to a new session and Run All.",
        )
        print("E2 PAUSED SAFELY. Save Version and continue in a new session.")
    elif E2_STATUS.get("status") == "completed":
        E2_MODEL_PATH = E2_RUN_DIR / "pvuad_last_model.pth"
        E2_CHECKPOINT_PATH = E2_RUN_DIR / "pvuad_last_checkpoint.pth"
        required = [E2_MODEL_PATH, E2_CHECKPOINT_PATH, E2_RUN_DIR / "metrics_history.csv"]
        missing = [str(path) for path in required if not path.is_file()]
        if missing:
            raise RuntimeError(f"Completed E2 is missing: {missing}")
        E2_COMPLETED_THIS_SESSION = True
        ACTIVE_STAGE = "E3"
else:
    if SELECTED_E2 is not None:
        E2_MODEL_PATH = SELECTED_E2.get("last_model")
        E2_CHECKPOINT_PATH = SELECTED_E2["checkpoint"]
        if E2_MODEL_PATH is None or not Path(E2_MODEL_PATH).is_file():
            raise FileNotFoundError(
                "LAST-only policy requires completed E2 pvuad_last_model.pth; "
                "best-model fallback is disabled."
            )


def create_teacher_cache(source_model, destination):
    source_model = Path(source_model)
    destination = Path(destination)
    if destination.is_file():
        return destination
    if not source_model.is_file():
        raise FileNotFoundError(source_model)
    try:
        payload = torch.load(source_model, map_location="cpu", weights_only=False)
    except TypeError:
        payload = torch.load(source_model, map_location="cpu")
    if isinstance(payload, dict) and "model" in payload:
        payload = payload["model"]
    elif isinstance(payload, dict) and "state_dict" in payload:
        payload = payload["state_dict"]
    if not isinstance(payload, dict):
        raise ValueError("E2 model file does not contain a state dictionary")
    teacher_state = {}
    for key, value in payload.items():
        clean = str(key).replace("module.", "")
        teacher_state[clean] = value.half() if torch.is_floating_point(value) else value
    temporary = destination.with_suffix(".pth.tmp")
    torch.save(teacher_state, temporary)
    os.replace(temporary, destination)
    del payload, teacher_state
    return destination


E3_STATUS = SELECTED_E3["status"] if SELECTED_E3 else {}
E3_RESUME_PATH = SELECTED_E3["checkpoint"] if SELECTED_E3 else None
TEACHER_CACHE = E3_RUN_DIR / "e2_teacher_model_fp16.pth"

if SELECTED_E3 is not None:
    copy_if_present(
        SELECTED_E3["root"], E3_RUN_DIR,
        [
            "metrics_history.csv", "pvuad_best_model.pth", "pvuad_last_model.pth",
            "e2_teacher_model_fp16.pth", "tta_best_metrics.csv", "tta_last_metrics.csv",
            "training_summary.json", "TRAINING_COMPLETE_BEFORE_TTA.json",
            "FINAL_TTA_FAILED.txt", "effective_test_command.txt",
        ],
    )


def e3_command(resume_path, student_model, prototype_checkpoint, teacher_cache):
    recipe = {
        "experiment": "E3-TDH", "stage": "final", "epochs": E3_EPOCHS,
        "global_batch": E3_GLOBAL_BATCH, "backbone_lr": E3_BASE_LR,
        "head_lr_multiplier": E3_HEAD_LR_MULTIPLIER,
        "tir_global_weight": 0.0,
        "tir_local_weight": 0.0,
        "feature_preserve_weight": FEATURE_PRESERVE_WEIGHT,
        "scenario_hard_weight": SCENARIO_HARD_WEIGHT,
        "initialization_origin": "MSMT17_TransReID_then_LAGPeR_S3CLIP_E2_epoch60",
        "dataset": "LAGPeR-S3CLIP", "protocol": "A2G_training_eval",
        "tir_distillation": False,
        "embedding_dim": 768,
        "backbone_stride": [16, 16],
    }
    manifest_path = make_stage_manifest(
        E3_RUN_DIR, "E3-TDH", recipe, resume_path=resume_path,
        e2_model_path=str(E2_MODEL_PATH) if E2_MODEL_PATH else None,
        e2_checkpoint_path=str(E2_CHECKPOINT_PATH) if E2_CHECKPOINT_PATH else None,
        teacher_model_path=str(teacher_cache),
    )
    opts = [
        "MODEL.DIST_TRAIN", "True",
        "MODEL.PRETRAIN_CHOICE", "none",
        "MODEL.PRETRAIN_PATH", str(PRETRAIN_PATH),
        "MODEL.STRIDE_SIZE", "[16, 16]",
        "MODEL.SIE_CAMERA", "False",
        "MODEL.SIE_VIEW", "False",
        "DATASETS.NAMES", "LAGPeR",
        "DATASETS.MODALITIES", "['RGB']",
        "DATASETS.ROOT_DIR", str(DATA_LINK_PARENT),
        "DATALOADER.NUM_WORKERS", str(NUM_WORKERS),
        "SOLVER.SEED", str(RANDOM_SEED),
        "SOLVER.MAX_EPOCHS", str(E3_EPOCHS),
        "SOLVER.IMS_PER_BATCH", str(E3_GLOBAL_BATCH),
        "SOLVER.BASE_LR", str(E3_BASE_LR),
        "SOLVER.WARMUP_EPOCHS", "1",
        "SOLVER.EVAL_PERIOD", str(E3_EPOCHS),
        "SOLVER.CHECKPOINT_PERIOD", "5",
        "INPUT.RE_PROB", "0.0",
        "TEST.IMS_PER_BATCH", "256",
        "TEST.RE_RANKING", "False",
        "TEST.TOP_K_EVAL", "0",
        "OUTPUT_DIR", str(E3_RUN_DIR),
        "PVUAD.EXPERIMENT", "E3-TDH",
        "PVUAD.STAGE", "final",
        "PVUAD.DEV_MODE", "False",
        "PVUAD.PAIRED_VIEW_SAMPLER", "True",
        "PVUAD.STRICT_PAIRED_DATA", "False",
        "PVUAD.GROUND_MAX_CAMID", str(GROUND_MAX_CAMID),
        "PVUAD.SYNC_PAIRED_AUG", "True",
        "PVUAD.VFPROCA", "True",
        "MODEL.PCA_LOSS_WEIGHT", "0.01",
        "PVUAD.VIEW_ALIGNMENT_WEIGHT", "0.5",
        "PVUAD.INVARIANT_ID_WEIGHT", "0.5",
        "PVUAD.SCENARIO_CLS_WEIGHT", "0.1",
        "PVUAD.TEST_INVARIANT_BLEND", "0.5",
        "PVUAD.LOCAL_CONSISTENCY", "False",
        "PVUAD.HARD_GPD", "False",
        "PVUAD.FINETUNE", "True",
        "PVUAD.WARMSTART_MODEL_PATH", str(student_model),
        "PVUAD.WARMSTART_PROTOTYPE_PATH", (
            str(prototype_checkpoint) if prototype_checkpoint else ""
        ),
        "PVUAD.TEACHER_MODEL_PATH", str(teacher_cache),
        "PVUAD.HEAD_LR_MULTIPLIER", str(float(E3_HEAD_LR_MULTIPLIER)),
        "PVUAD.TIR_DISTILLATION", "False",
        "PVUAD.TIR_GLOBAL_WEIGHT", "0.0",
        "PVUAD.TIR_LOCAL_WEIGHT", "0.0",
        "PVUAD.FEATURE_PRESERVE_WEIGHT", str(float(FEATURE_PRESERVE_WEIGHT)),
        "PVUAD.SCENARIO_HARD_GPD", "True",
        "PVUAD.SCENARIO_HARD_WEIGHT", str(float(SCENARIO_HARD_WEIGHT)),
        "PVUAD.SCENARIO_HARD_TOPK", str(int(SCENARIO_HARD_TOPK)),
        "PVUAD.SCENARIO_HARD_MARGIN", str(float(SCENARIO_HARD_MARGIN)),
        "PVUAD.SCENARIO_HARD_TAU", str(float(SCENARIO_HARD_TAU)),
        "PVUAD.SCENARIO_HARD_START_EPOCH", str(int(SCENARIO_HARD_START_EPOCH)),
        "PVUAD.SCENARIO_BANK_MOMENTUM", "0.2",
        "PVUAD.FINAL_FLIP_TTA", "False",
        "PVUAD.SAVE_LAST_EVERY", "5",
        "PVUAD.RUN_MANIFEST", str(manifest_path),
        "PVUAD.SESSION_DEADLINE_UNIX", str(float(SESSION_DEADLINE_UNIX)),
        "PVUAD.STOP_RESERVE_MINUTES", str(float(SESSION_STOP_RESERVE_MINUTES)),
        "PVUAD.CHECKPOINT_EVERY_MINUTES", str(float(CHECKPOINT_EVERY_MINUTES)),
        "PVUAD.TIME_CHECK_EVERY_STEPS", str(int(TIME_CHECK_EVERY_STEPS)),
        "PVUAD.MIN_EVAL_REMAINING_MINUTES", str(float(MIN_EVAL_REMAINING_MINUTES)),
        "PVUAD.STRICT_RESUME_CONFIG", "True",
    ]
    if resume_path:
        opts += ["PVUAD.RESUME_PATH", str(resume_path)]
    return [
        sys.executable, "-m", "torch.distributed.run", "--standalone",
        f"--nproc_per_node={NUM_GPUS}", "train.py", "--config_file",
        "configs/PVUAD.yml", *opts,
    ]


can_enter_e3 = (
    ACTIVE_STAGE == "E3"
    and (E3_STATUS.get("status") != "completed")
    and RUN_TRAINING
)
if (
    can_enter_e3
    and E2_COMPLETED_THIS_SESSION
    and ALWAYS_START_E3_IN_NEW_SESSION
):
    write_pipeline_state(
        "e2_completed_e3_pending", "E3",
        e2_status=E2_STATUS,
        remaining_minutes=remaining_minutes(),
        next_action=(
            "Kaggle will save this completed E2 output. Attach it to a new "
            "session and use Save & Run All; E3 will start automatically."
        ),
    )
    print(
        "E2 COMPLETE. This Kaggle version now ends cleanly by design. "
        "Attach its output to the next session; E3 will start there."
    )
    can_enter_e3 = False
elif can_enter_e3 and remaining_minutes() < MIN_E3_START_REMAINING_MINUTES:
    pending_status = "e3_paused" if E3_RESUME_PATH else "e2_completed_e3_pending"
    write_pipeline_state(
        pending_status, "E3",
        e2_status=E2_STATUS,
        remaining_minutes=remaining_minutes(),
        next_action="Save Version; attach this output and Run All in a new session.",
    )
    print(
        "Only {:.1f} minutes remain. E3 will start/resume in the next "
        "session.".format(remaining_minutes())
    )
    can_enter_e3 = False

if can_enter_e3:
    if E3_RESUME_PATH is not None:
        teacher_source = (
            SELECTED_E3["root"] / "e2_teacher_model_fp16.pth"
        )
        if not teacher_source.is_file() and E2_MODEL_PATH is not None:
            teacher_source = E2_MODEL_PATH
        if not TEACHER_CACHE.is_file():
            if teacher_source.name == "e2_teacher_model_fp16.pth":
                shutil.copy2(teacher_source, TEACHER_CACHE)
            else:
                create_teacher_cache(teacher_source, TEACHER_CACHE)
        student_model = TEACHER_CACHE
        prototype_checkpoint = None
    else:
        if E2_MODEL_PATH is None or not Path(E2_MODEL_PATH).is_file():
            raise FileNotFoundError(
                "E3 needs a completed E2 final model. Attach the "
                "previous combined/E2 output or use E2_COMPLETED_DIR_OVERRIDE."
            )
        if E2_CHECKPOINT_PATH is None or not Path(E2_CHECKPOINT_PATH).is_file():
            raise FileNotFoundError(
                "E3 needs E2 pvuad_last_checkpoint.pth for prototype warm-start."
            )
        create_teacher_cache(E2_MODEL_PATH, TEACHER_CACHE)
        student_model = E2_MODEL_PATH
        prototype_checkpoint = E2_CHECKPOINT_PATH

    print("Frozen E2 teacher:", TEACHER_CACHE)
    write_pipeline_state(
        "e3_running", "E3",
        e2_status=E2_STATUS,
        e3_resume_path=str(E3_RESUME_PATH) if E3_RESUME_PATH else None,
    )
    E3_STATUS = run_command_checked(
        e3_command(
            E3_RESUME_PATH, student_model, prototype_checkpoint, TEACHER_CACHE
        ),
        E3_RUN_DIR,
        "E3-TDH (LAGPeR RGB adaptation)",
    )
    if E3_STATUS.get("status") in {"paused_time_limit", "running_checkpoint"}:
        write_pipeline_state(
            "e3_paused", "E3", e2_status=E2_STATUS, e3_status=E3_STATUS,
            next_action="Attach this output to a new session and Run All.",
        )
        print("E3 PAUSED SAFELY. Save Version and continue in a new session.")


def run_final_tta(model_path, protocol):
    """Evaluate one LAGPeR protocol using ONLY E3 last model."""
    protocol = str(protocol).upper()
    if protocol not in FINAL_PROTOCOLS:
        raise ValueError(protocol)

    metrics_filename = f"tta_{protocol}_last_metrics.csv"
    metrics_path = E3_RUN_DIR / metrics_filename
    if metrics_path.is_file():
        print(f"{protocol}: already evaluated -> {metrics_path}")
        return True

    test_opts = [
        "MODEL.DIST_TRAIN", "False",
        "MODEL.PRETRAIN_CHOICE", "none",
        "MODEL.PRETRAIN_PATH", str(PRETRAIN_PATH),
        "MODEL.STRIDE_SIZE", "[16, 16]",
        "MODEL.SIE_CAMERA", "False",
        "MODEL.SIE_VIEW", "False",
        "DATASETS.NAMES", "LAGPeR",
        "DATASETS.MODALITIES", "['RGB']",
        "DATASETS.ROOT_DIR", str(DATA_LINK_PARENT),
        "DATALOADER.NUM_WORKERS", str(NUM_WORKERS),
        "TEST.IMS_PER_BATCH", "256",
        "TEST.RE_RANKING", "False",
        "TEST.TOP_K_EVAL", "0",
        "TEST.WEIGHT", str(model_path),
        "OUTPUT_DIR", str(E3_RUN_DIR),
        "PVUAD.EXPERIMENT", "E3-TDH",
        "PVUAD.STAGE", "final",
        "PVUAD.DEV_MODE", "False",
        "PVUAD.PAIRED_VIEW_SAMPLER", "True",
        "PVUAD.STRICT_PAIRED_DATA", "False",
        "PVUAD.GROUND_MAX_CAMID", str(GROUND_MAX_CAMID),
        "PVUAD.VFPROCA", "True",
        "PVUAD.TEST_INVARIANT_BLEND", "0.5",
        "PVUAD.TIR_DISTILLATION", "False",
        "PVUAD.FEATURE_PRESERVE_WEIGHT", "0.0",
        "PVUAD.SCENARIO_HARD_GPD", "True",
        "PVUAD.FINAL_FLIP_TTA", "True",
        "PVUAD.TTA_METRICS_FILENAME", metrics_filename,
    ]
    test_command = [
        sys.executable, "test.py", "--config_file", "configs/PVUAD.yml",
        *test_opts,
    ]
    (E3_RUN_DIR / f"effective_test_command_{protocol}.txt").write_text(
        shlex.join(test_command) + "\n", encoding="utf-8"
    )

    test_env = environment.copy()
    test_env["CUDA_VISIBLE_DEVICES"] = "0"
    test_env["PVUAD_LAGPER_PROTOCOL"] = protocol
    test_env["PVUAD_LAGPER_MARKET_EVAL"] = "1"

    print(f"Running LAGPeR-S3CLIP {protocol} with E3 LAST model + flip-TTA")
    print(" ", model_path)
    result = subprocess.run(
        test_command, cwd=UAD_DIR, env=test_env, check=False
    )
    if result.returncode != 0 or not metrics_path.is_file():
        warning = (
            f"{protocol} final TTA failed; E3 last checkpoint remains valid. "
            f"returncode={result.returncode}."
        )
        (E3_RUN_DIR / f"FINAL_TTA_FAILED_{protocol}.txt").write_text(
            warning + "\n", encoding="utf-8"
        )
        print("WARNING:", warning)
        return False
    return True


E3_COMPLETED = E3_STATUS.get("status") == "completed"
if E3_COMPLETED:
    if SELECTED_E3 is not None:
        copy_if_present(
            SELECTED_E3["root"], E3_RUN_DIR,
            [
                "pvuad_last_checkpoint.pth", "pvuad_last_model.pth",
                "session_status.json", "training_summary.json", "metrics_history.csv",
                "tta_A2G_last_metrics.csv", "tta_G2A_last_metrics.csv",
                "tta_G2AG_last_metrics.csv",
            ],
        )

    # LAST-only policy: pvuad_best_model.pth is intentionally irrelevant.
    last_model = E3_RUN_DIR / "pvuad_last_model.pth"
    last_checkpoint = E3_RUN_DIR / "pvuad_last_checkpoint.pth"
    if not last_model.is_file():
        raise FileNotFoundError("Completed E3 is missing pvuad_last_model.pth")
    if not last_checkpoint.is_file():
        raise FileNotFoundError("Completed E3 is missing pvuad_last_checkpoint.pth")

    protocol_status = {}
    for protocol in FINAL_PROTOCOLS:
        metrics_path = E3_RUN_DIR / f"tta_{protocol}_last_metrics.csv"
        ok = metrics_path.is_file()
        if RUN_FINAL_FLIP_TTA and not ok:
            if remaining_minutes() >= MIN_TTA_START_REMAINING_MINUTES:
                ok = run_final_tta(last_model, protocol)
            else:
                print(
                    f"E3 complete but only {remaining_minutes():.1f} min remain; "
                    f"{protocol} TTA will continue next session."
                )
        protocol_status[protocol] = bool(ok or not RUN_FINAL_FLIP_TTA)

    all_done = all(protocol_status.values())
    final_status = "completed" if all_done else "e3_complete_tta_pending"
    write_pipeline_state(
        final_status,
        "DONE" if all_done else "TTA",
        e2_status=E2_STATUS,
        e3_status=E3_STATUS,
        checkpoint_policy="last_only",
        e2_to_e3_weights="E2/pvuad_last_model.pth",
        e2_to_e3_prototype="E2/pvuad_last_checkpoint.pth:prototype_bank",
        final_test_weights="E3/pvuad_last_model.pth",
        final_protocols=protocol_status,
        next_action=(
            None if all_done
            else "Attach this output and Run All; training is skipped and pending protocol TTA resumes."
        ),
    )
elif ACTIVE_STAGE == "E3" and not can_enter_e3 and not RUN_TRAINING:
    write_pipeline_state("dry_run", "E3")


Frozen E2 teacher: /kaggle/working/pvuad_runs/lagper_s3clip_e2_60_e3_25_transreid768_b24_lastonly_v5_metricsindentfix/e3_tdh/e2_teacher_model_fp16.pth

Launching E3-TDH (LAGPeR RGB adaptation)
/usr/bin/python3 -m torch.distributed.run --standalone --nproc_per_node=2 train.py --config_file configs/PVUAD.yml MODEL.DIST_TRAIN True MODEL.PRETRAIN_CHOICE none MODEL.PRETRAIN_PATH /kaggle/working/vit_transreid_msmt.pth MODEL.STRIDE_SIZE '[16, 16]' MODEL.SIE_CAMERA False MODEL.SIE_VIEW False DATASETS.NAMES LAGPeR DATASETS.MODALITIES '['"'"'RGB'"'"']' DATASETS.ROOT_DIR /kaggle/working/pvuad_lagper_data DATALOADER.NUM_WORKERS 4 SOLVER.SEED 1234 SOLVER.MAX_EPOCHS 25 SOLVER.IMS_PER_BATCH 24 SOLVER.BASE_LR 0.0001 SOLVER.WARMUP_EPOCHS 1 SOLVER.EVAL_PERIOD 25 SOLVER.CHECKPOINT_PERIOD 5 INPUT.RE_PROB 0.0 TEST.IMS_PER_BATCH 256 TEST.RE_RANKING False TEST.TOP_K_EVAL 0 OUTPUT_DIR /kaggle/working/pvuad_runs/lagper_s3clip_e2_60_e3_25_transreid768_b24_lastonly_v5_metricsindentfix/e3_tdh PVUAD.EXPERIMENT E3-

[W917 19:04:55.510527909 socket.cpp:207] [c10d] The hostname of the client socket cannot be retrieved. err=-3


2026-09-17 19:05:07,942 transreid INFO: Saving model in the path :/kaggle/working/pvuad_runs/lagper_s3clip_e2_60_e3_25_transreid768_b24_lastonly_v5_metricsindentfix/e3_tdh
2026-09-17 19:05:07,942 transreid INFO: Namespace(config_file='configs/PVUAD.yml', opts=['MODEL.DIST_TRAIN', 'True', 'MODEL.PRETRAIN_CHOICE', 'none', 'MODEL.PRETRAIN_PATH', '/kaggle/working/vit_transreid_msmt.pth', 'MODEL.STRIDE_SIZE', '[16, 16]', 'MODEL.SIE_CAMERA', 'False', 'MODEL.SIE_VIEW', 'False', 'DATASETS.NAMES', 'LAGPeR', 'DATASETS.MODALITIES', "['RGB']", 'DATASETS.ROOT_DIR', '/kaggle/working/pvuad_lagper_data', 'DATALOADER.NUM_WORKERS', '4', 'SOLVER.SEED', '1234', 'SOLVER.MAX_EPOCHS', '25', 'SOLVER.IMS_PER_BATCH', '24', 'SOLVER.BASE_LR', '0.0001', 'SOLVER.WARMUP_EPOCHS', '1', 'SOLVER.EVAL_PERIOD', '25', 'SOLVER.CHECKPOINT_PERIOD', '5', 'INPUT.RE_PROB', '0.0', 'TEST.IMS_PER_BATCH', '256', 'TEST.RE_RANKING', 'False', 'TEST.TOP_K_EVAL', '0', 'OUTPUT_DIR', '/kaggle/working/pvuad_runs/lagper_s3clip_e2_60_e3_25_tr

[W917 19:05:07.409303493 socket.cpp:207] [c10d] The hostname of the client socket cannot be retrieved. err=-3
[W917 19:05:07.418134319 socket.cpp:207] [c10d] The hostname of the client socket cannot be retrieved. err=-3


=> LAGPeR loaded (A2G)
   train       : /kaggle/working/pvuad_lagper_data/LAGPeR/bounding_box_train
   query       : /kaggle/working/pvuad_lagper_data/LAGPeR/query_aerial
   gallery dirs: ['/kaggle/working/pvuad_lagper_data/LAGPeR/bounding_box_test_ground', '/kaggle/working/pvuad_lagper_data/LAGPeR/bounding_box_gallery']
   gallery noise: 134
=> LAGPeR loaded (A2G)
   train       : /kaggle/working/pvuad_lagper_data/LAGPeR/bounding_box_train
   query       : /kaggle/working/pvuad_lagper_data/LAGPeR/query_aerial
   gallery dirs: ['/kaggle/working/pvuad_lagper_data/LAGPeR/bounding_box_test_ground', '/kaggle/working/pvuad_lagper_data/LAGPeR/bounding_box_gallery']
   gallery noise: 134
Dataset statistics:
  ----------------------------------------
  subset   | # ids | # images | # cameras
  ----------------------------------------
  train    |  2708 |    40770 |        12
  query    |  1523 |     3046 |         3
  gallery  |  1524 |    15533 |        12
  ----------------------------------

/usr/local/lib/python3.12/dist-packages/torch/distributed/c10d_logger.py:83: UserWarning: barrier(): using the device under current context. You can specify `device_id` in `init_process_group` to mute this warning.
  return func(*args, **kwargs)


2026-09-17 19:36:31,992 transreid.train INFO: Epoch[5] Iteration[1350/2281] Loss 0.869 Acc 0.982 ID 0.369 TRI 0.014 Align 0.046 GPD 0.011 TIR-G 0.000 TIR-L 0.000 Keep 0.022 S-Hard 0.006 LR 9.05e-05
2026-09-17 19:36:31,992 transreid.train INFO: Epoch[5] Iteration[1350/2281] Loss 0.860 Acc 0.984 ID 0.364 TRI 0.014 Align 0.046 GPD 0.011 TIR-G 0.000 TIR-L 0.000 Keep 0.022 S-Hard 0.005 LR 9.05e-05
2026-09-17 19:36:40,566 transreid.train INFO: Epoch[5] Iteration[1400/2281] Loss 0.859 Acc 0.984 ID 0.363 TRI 0.014 Align 0.046 GPD 0.011 TIR-G 0.000 TIR-L 0.000 Keep 0.022 S-Hard 0.005 LR 9.05e-05
2026-09-17 19:36:40,566 transreid.train INFO: Epoch[5] Iteration[1400/2281] Loss 0.869 Acc 0.982 ID 0.369 TRI 0.014 Align 0.046 GPD 0.011 TIR-G 0.000 TIR-L 0.000 Keep 0.022 S-Hard 0.006 LR 9.05e-05
2026-09-17 19:36:49,200 transreid.train INFO: Epoch[5] Iteration[1450/2281] Loss 0.872 Acc 0.981 ID 0.370 TRI 0.014 Align 0.046 GPD 0.011 TIR-G 0.000 TIR-L 0.000 Keep 0.022 S-Hard 0.006 LR 9.05e-05
2026-09-17

,stage,status,reason,epoch,next_iteration,epoch_complete,global_step,best_mAP,remaining_minutes
0,E3-TDH (LAGPeR RGB adaptation),completed,training_completed,25,0,True,57025,27.169722,370.5


Running LAGPeR-S3CLIP A2G with E3 LAST model + flip-TTA
  /kaggle/working/pvuad_runs/lagper_s3clip_e2_60_e3_25_transreid768_b24_lastonly_v5_metricsindentfix/e3_tdh/pvuad_last_model.pth
2026-09-17 21:54:09,055 transreid INFO: Namespace(config_file='configs/PVUAD.yml', opts=['MODEL.DIST_TRAIN', 'False', 'MODEL.PRETRAIN_CHOICE', 'none', 'MODEL.PRETRAIN_PATH', '/kaggle/working/vit_transreid_msmt.pth', 'MODEL.STRIDE_SIZE', '[16, 16]', 'MODEL.SIE_CAMERA', 'False', 'MODEL.SIE_VIEW', 'False', 'DATASETS.NAMES', 'LAGPeR', 'DATASETS.MODALITIES', "['RGB']", 'DATASETS.ROOT_DIR', '/kaggle/working/pvuad_lagper_data', 'DATALOADER.NUM_WORKERS', '4', 'TEST.IMS_PER_BATCH', '256', 'TEST.RE_RANKING', 'False', 'TEST.TOP_K_EVAL', '0', 'TEST.WEIGHT', '/kaggle/working/pvuad_runs/lagper_s3clip_e2_60_e3_25_transreid768_b24_lastonly_v5_metricsindentfix/e3_tdh/pvuad_last_model.pth', 'OUTPUT_DIR', '/kaggle/working/pvuad_runs/lagper_s3clip_e2_60_e3_25_transreid768_b24_lastonly_v5_metricsindentfix/e3_tdh', 'PVUAD.EXP

## 6. Tổng hợp kết quả paper-ready

Report phân biệt rõ E2/E3 epoch cuối và ba final protocol. Không đọc/không chọn `pvuad_best_model.pth`.

In [7]:
def metrics_history_last(run_dir, label):
    path = Path(run_dir) / "metrics_history.csv"
    if not path.is_file():
        return None
    frame = pd.read_csv(path)
    required = {"experiment", "stage", "epoch", "mAP", "Rank-1", "Rank-5", "Rank-10"}
    if frame.empty or not required.issubset(frame.columns):
        return None
    frame = frame.drop_duplicates(
        subset=["experiment", "stage", "epoch"], keep="last"
    ).sort_values("epoch")
    frame.to_csv(path, index=False)
    row = frame.iloc[-1]
    return {
        "method": label,
        "protocol": "A2G epoch-end diagnostic",
        "epoch": int(row["epoch"]),
        "mAP": float(row["mAP"]),
        "Rank-1": float(row["Rank-1"]),
        "Rank-5": float(row["Rank-5"]),
        "Rank-10": float(row["Rank-10"]),
        "checkpoint_used_for_next_step": "last",
    }

def tta_metrics(protocol):
    path = E3_RUN_DIR / f"tta_{protocol}_last_metrics.csv"
    if not path.is_file():
        return None
    frame = pd.read_csv(path)
    if frame.empty:
        return None
    row = frame.iloc[-1]
    return {
        "method": "E3 LAST + horizontal-flip TTA",
        "protocol": protocol,
        "epoch": E3_EPOCHS,
        "mAP": float(row["mAP"]),
        "Rank-1": float(row["Rank-1"]),
        "Rank-5": float(row["Rank-5"]),
        "Rank-10": float(row["Rank-10"]),
        "checkpoint_used_for_next_step": "final_test",
    }

state = read_json(PIPELINE_STATE_PATH)
print("Pipeline status:", state.get("status"))
print("Active stage   :", state.get("active_stage"))
print("Next action    :", state.get("next_action"))
print("Dataset audit  :")
display(pd.DataFrame([{
    "dataset": DATASET_AUDIT["dataset"],
    "train_ids": DATASET_AUDIT["train_ids"],
    "train_images": DATASET_AUDIT["train_images"],
    "A2G_q/g": f"{DATASET_AUDIT['A2G']['query_images']}/{DATASET_AUDIT['A2G']['gallery_images']}",
    "G2A_q/g": f"{DATASET_AUDIT['G2A']['query_images']}/{DATASET_AUDIT['G2A']['gallery_images']}",
    "G2AG_q/g": f"{DATASET_AUDIT['G2AG']['query_images']}/{DATASET_AUDIT['G2AG']['gallery_images']}",
}]))

init_audit_path = PIPELINE_ROOT / "transreid_trainable_init_audit.json"
if init_audit_path.is_file():
    print("TransReID initialization audit:")
    display(pd.DataFrame([read_json(init_audit_path)]))

comparison = []
e2_last = metrics_history_last(E2_RUN_DIR, f"E2 LAST epoch {E2_EPOCHS}")
e3_last = metrics_history_last(E3_RUN_DIR, f"E3 LAST epoch {E3_EPOCHS} (no TTA)")
if e2_last:
    comparison.append(e2_last)
if e3_last:
    comparison.append(e3_last)

for protocol in FINAL_PROTOCOLS:
    result = tta_metrics(protocol)
    if result:
        comparison.append(result)

comparison_frame = pd.DataFrame(comparison)
comparison_path = PIPELINE_ROOT / "combined_results.csv"
comparison_frame.to_csv(comparison_path, index=False)
if not comparison_frame.empty:
    display(comparison_frame.round(4))

checkpoint_policy = {
    "policy": "last_only",
    "E2_fixed_horizon": E2_EPOCHS,
    "E3_fixed_horizon": E3_EPOCHS,
    "E2_to_E3_weights": "e2_final/pvuad_last_model.pth",
    "E2_to_E3_prototype_bank": "e2_final/pvuad_last_checkpoint.pth:prototype_bank",
    "E3_to_final_tests": "e3_tdh/pvuad_last_model.pth",
    "intermediate_checkpoint_selection": False,
    "E2_eval_period": E2_EPOCHS,
    "E3_eval_period": E3_EPOCHS,
    "final_protocols": list(FINAL_PROTOCOLS),
    "final_flip_tta": bool(RUN_FINAL_FLIP_TTA),
    "reranking": False,
    "lagper_eval_junk_rule": "remove same PID + same camera only",
}
(PIPELINE_ROOT / "final_checkpoint_policy.json").write_text(
    json.dumps(checkpoint_policy, indent=2), encoding="utf-8"
)
print("Checkpoint policy:", checkpoint_policy)

REPORT_ZIP = (
    Path("/kaggle/working")
    / "pvuad_lagper_s3clip_e2_60_e3_25_transreid_reports.zip"
)
if REPORT_ZIP.exists():
    REPORT_ZIP.unlink()

report_suffixes = {".csv", ".json", ".txt", ".log", ".yml", ".yaml"}
with zipfile.ZipFile(
    REPORT_ZIP, "w", compression=zipfile.ZIP_DEFLATED
) as archive:
    for path in sorted(PIPELINE_ROOT.rglob("*")):
        if path.is_file() and path.suffix.lower() in report_suffixes:
            archive.write(path, arcname=str(path.relative_to(PIPELINE_ROOT)))

    extras = [
        Path("/kaggle/working/lagper_s3clip_dataset_audit.json"),
        Path("/kaggle/working/lagper_s3clip_split_audit.csv"),
        Path("/kaggle/working/lagper_exp_txt_preview.json"),
        CAMERA_MAP_PATH,
    ]
    for path in extras:
        if path.is_file():
            archive.write(path, arcname=path.name)

    patch_manifest = UAD_DIR / "PVUAD_patch_manifest.json"
    if patch_manifest.is_file():
        archive.write(patch_manifest, arcname="PVUAD_patch_manifest.json")

print("Report zip:", REPORT_ZIP)

if all(tta_metrics(protocol) is not None for protocol in FINAL_PROTOCOLS):
    final_table = comparison_frame[
        (comparison_frame["method"] == "E3 LAST + horizontal-flip TTA")
    ][["protocol", "mAP", "Rank-1", "Rank-5", "Rank-10"]]
    print("FINAL LAGPeR-S3CLIP:")
    display(final_table.round(4))


Pipeline status: completed
Active stage   : DONE
Next action    : None
Dataset audit  :


,dataset,train_ids,train_images,A2G_q/g,G2A_q/g,G2AG_q/g
0,LAGPeR-S3CLIP,2708,40770,3046/15533,3046/7717,3046/20204


TransReID initialization audit:


,checkpoint,loaded_tensor_count,target_tensor_count,mismatch_count,resized_pos_embed,source_sie_loaded,source_jpm_local_branches_loaded,global_jpm_branch_used_for_target_final_block,global_branch_mapped_tensor_count,embedding_dim,...,batchnorm_train_mode_checked,init_qkv_max_abs,global_final_qkv_max_abs,qkv_grad_norm,patch_grad_norm,backbone_all_trainable,source_sie_used,source_jpm_local_branches_used,source_jpm_global_final_branch_used,target_stride
0,/kaggle/working/vit_transreid_msmt.pth,152,152,0,"{'source_shape': [1, 211, 768], 'target_shape'...",False,False,True,14,768,...,True,0.0,0.0,0.004461,0.022849,True,False,False,True,"[16, 16]"


,method,protocol,epoch,mAP,Rank-1,Rank-5,Rank-10,checkpoint_used_for_next_step
0,E2 LAST epoch 60,A2G epoch-end diagnostic,60,26.9829,37.8858,47.2095,52.7577,last
1,E3 LAST epoch 25 (no TTA),A2G epoch-end diagnostic,25,27.1697,38.2141,47.1766,52.8562,last
2,E3 LAST + horizontal-flip TTA,A2G,25,27.7150,38.0827,52.9547,61.0965,final_test
3,E3 LAST + horizontal-flip TTA,G2A,25,30.8546,31.7137,44.4517,51.8056,final_test
4,E3 LAST + horizontal-flip TTA,G2AG,25,18.0829,21.8976,33.4865,41.0046,final_test


Checkpoint policy: {'policy': 'last_only', 'E2_fixed_horizon': 60, 'E3_fixed_horizon': 25, 'E2_to_E3_weights': 'e2_final/pvuad_last_model.pth', 'E2_to_E3_prototype_bank': 'e2_final/pvuad_last_checkpoint.pth:prototype_bank', 'E3_to_final_tests': 'e3_tdh/pvuad_last_model.pth', 'intermediate_checkpoint_selection': False, 'E2_eval_period': 60, 'E3_eval_period': 25, 'final_protocols': ['A2G', 'G2A', 'G2AG'], 'final_flip_tta': True, 'reranking': False, 'lagper_eval_junk_rule': 'remove same PID + same camera only'}
Report zip: /kaggle/working/pvuad_lagper_s3clip_e2_60_e3_25_transreid_reports.zip
FINAL LAGPeR-S3CLIP:


,protocol,mAP,Rank-1,Rank-5,Rank-10
2,A2G,27.7150,38.0827,52.9547,61.0965
3,G2A,30.8546,31.7137,44.4517,51.8056
4,G2AG,18.0829,21.8976,33.4865,41.0046


## Cách chạy trên Kaggle

1. Chọn **GPU T4 ×2**.
2. Attach dataset **LAGPeR-S3CLIP**. Không cần đúng một Kaggle path cố định:
   notebook tự tìm root có đủ `bounding_box_train`, `query_aerial`, `query_ground`,
   `bounding_box_test_aerial`, `bounding_box_test_ground`, `bounding_box_gallery`.
3. Phiên đầu bật Internet để tải official `vit_transreid_msmt.pth`, hoặc attach
   checkpoint đó làm Kaggle Input.
4. Chọn **Save Version → Save & Run All**.
5. Nếu notebook dừng an toàn vì giới hạn phiên, phiên sau attach **output của chính
   version ngay trước đó** làm Input rồi chạy lại **Save & Run All**.
6. E2 train cố định **60 epochs**. E3 nhận **LAST E2 model + prototype bank từ
   LAST E2 checkpoint**, rồi train cố định **25 epochs**.
7. Final inference dùng **LAST E3 model + horizontal-flip TTA**, không dùng
   `pvuad_best_model.pth` để chọn checkpoint.
8. Notebook tự chạy ba protocol: **A→G**, **G→A**, và **G→A+G**, rồi xuất:
   `/kaggle/working/pvuad_lagper_s3clip_e2_60_e3_25_transreid_reports.zip`.
9. Nếu cell audit báo camera-view inference không khớp, điền
   `AERIAL_CAMERA_IDS_OVERRIDE` ở cell Settings bằng **7 raw aerial camera IDs**
   của LAGPeR-S3CLIP rồi chạy lại từ đầu.
